# Text-to-SVG: SFT → Strict-Adapter DPO Mining → DPO → Multi-Candidate Inference

This notebook is a stricter, safer follow-up to v8 for the NYU Text-to-SVG competition.

What changed from v8:
- **Adapter/base-model loading is now strict**: if an adapter has its own `base_model_name_or_path`, runtime uses that exact base model for mining/inference/training instead of silently swapping to another repo alias.
- **Missing adapter keys are treated as a hard failure** during adapter attachment instead of a warning that gets ignored.
- Added a **generation smoke test gate** before DPO mining so broken adapter loads fail immediately, not after hundreds of prompts.
- DPO mining now prints the **real generation error excerpt** and aborts early after a short streak of generation failures.
- Added safer CUDA cleanup so a poisoned CUDA context does not cause misleading secondary crashes in `empty_cache()`.
- Kept the repaired SVG sanitization, processed-based heartbeat logs, JSONL tracing, and hybrid DPO option from v8.

Recommended use when SFT is already finished:
1. Set `RUN_SFT = False`
2. Set `RUN_BUILD_DPO = True`
3. Set `RUN_DPO = True`
4. During debugging, keep `RUN_INFERENCE = False`
5. Restart runtime before rerunning after any CUDA device-side assert


## 0. Setup & Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/v6'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/outputs', exist_ok=True)
print(f'PROJECT_DIR = {PROJECT_DIR}')
print(f'Contents: {os.listdir(PROJECT_DIR)}')


In [ ]:
!pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas pyarrow lxml cairosvg pillow scikit-image mergekit

import importlib.util, subprocess, sys
print('[SETUP] Removing llm-blender if present because TRL DPO training does not require judges, and an installed llm-blender can break TRL imports with newer transformers.')
subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'llm-blender'])


## 1. Configuration & Switches

In [ ]:
# ══════════════════════════════════════════════
# PIPELINE SWITCHES — toggle which stages to run
# ══════════════════════════════════════════════
RUN_SFT = False
RUN_BUILD_DPO = False
RUN_DPO = False
RUN_INFERENCE = True


In [ ]:
from __future__ import annotations

import gc
import io
import json
import math
import os
import random
import re
import subprocess
import shutil
import time
import traceback
import hashlib
import xml.etree.ElementTree as StdET
from collections import Counter, OrderedDict, defaultdict, deque
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from lxml import etree as LET
from PIL import Image
from skimage.feature import canny
from skimage.metrics import structural_similarity as ssim

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'Torch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# ══════════════════════════════════════════════
# Competition constraints
# ══════════════════════════════════════════════
ALLOWED_TAGS = {
    'svg', 'g', 'path', 'rect', 'circle', 'ellipse', 'line', 'polyline',
    'polygon', 'defs', 'use', 'symbol', 'clipPath', 'mask',
    'linearGradient', 'radialGradient', 'stop', 'text', 'tspan',
    'title', 'desc', 'style', 'pattern', 'marker', 'filter',
}
MAX_SVG_CHARS = 16000
MAX_PATHS = 256
SVG_NS = 'http://www.w3.org/2000/svg'
SVG_REGEX = re.compile(r'<svg[\s\S]*?</svg>', re.IGNORECASE)
FLOAT_REGEX = re.compile(r'(?<![A-Za-z0-9_.-])-?\d+\.\d+')
COLOR_WORDS = [
    'red', 'blue', 'green', 'yellow', 'orange', 'purple', 'black', 'white',
    'pink', 'brown', 'gray', 'grey', 'gold', 'silver', 'cyan', 'teal',
]
DANGEROUS_ATTR_PREFIXES = ('on',)
REF_ATTRS = {'href', 'xlink:href'}
PRIORITY_TAGS = {
    'defs': 0.1, 'title': 0.1, 'desc': 0.1, 'style': 0.1,
    'path': 1.0, 'rect': 0.9, 'circle': 0.9, 'ellipse': 0.9, 'line': 0.8,
    'polyline': 0.8, 'polygon': 0.8, 'text': 0.7, 'tspan': 0.7, 'g': 0.2,
    'linearGradient': 0.2, 'radialGradient': 0.2, 'stop': 0.1,
    'pattern': 0.2, 'marker': 0.1, 'clipPath': 0.2, 'mask': 0.2,
    'filter': 0.2, 'symbol': 0.2, 'use': 0.4, 'svg': 10.0,
}

In [ ]:
# ══════════════════════════════════════════════
# Config — Google Drive paths
# ══════════════════════════════════════════════
LORA_TARGET_MODULES = (
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'up_proj', 'down_proj',
)

@dataclass
class Config:
    # ---- paths ----
    project_dir: str = PROJECT_DIR
    train_csv: str = field(default_factory=lambda: f'{PROJECT_DIR}/train.csv')
    test_csv: str = field(default_factory=lambda: f'{PROJECT_DIR}/test.csv')
    output_dir: str = field(default_factory=lambda: f'{PROJECT_DIR}/outputs')

    # ---- model ----
    model_name_train: str = 'unsloth/Qwen3.5-2B'
    model_name_infer: str = 'unsloth/Qwen3.5-2B'
    max_seq_length: int = 2048
    adapter_base_model_override: Optional[str] = None
    strict_adapter_base_match: bool = True
    strict_adapter_missing_keys: bool = True
    strict_adapter_signature_for_trainable: bool = True
    adapter_load_strategy: str = 'unsloth_direct_then_runtime_patch'
    adapter_direct_load_first: bool = True
    adapter_allow_manual_attach_fallback: bool = False

    adapter_runtime_cache_dir: str = field(default_factory=lambda: f'{PROJECT_DIR}/outputs/adapter_runtime_cache')
    adapter_repair_target_modules_from_checkpoint: bool = True
    adapter_repair_fail_if_no_checkpoint_modules: bool = True
    adapter_repair_allow_subset_target_modules: bool = True

    # ---- system prompt ----
    system_prompt: str = (
        'You are an SVG generator. '
        'Output exactly one valid SVG and nothing else. '
        'No markdown. No explanation. '
        'Canvas must be 256x256 with viewBox 0 0 256 256. '
        'Use only allowed SVG tags. '
        'Keep the drawing visually faithful to the prompt. '
        'Return compact SVG that ends with </svg>.'
    )

    # ---- SFT / LoRA ----
    lora_r: int = 32
    lora_alpha: int = 64
    lora_dropout: float = 0.05
    lora_target_modules: Tuple[str, ...] = LORA_TARGET_MODULES
    learning_rate_sft: float = 1.5e-4
    num_train_epochs_sft: float = 2.0
    per_device_train_batch_size_sft: int = 32
    gradient_accumulation_steps_sft: int = 1
    warmup_ratio_sft: float = 0.05
    weight_decay_sft: float = 0.01
    max_grad_norm_sft: float = 0.3
    eval_fraction: float = 0.03
    logging_steps: int = 20
    save_steps: int = 500
    eval_steps: int = 500
    save_total_limit: int = 2

    # ---- DPO training ----
    learning_rate_dpo: float = 1e-5
    num_train_epochs_dpo: float = 1.0
    per_device_train_batch_size_dpo: int = 2
    gradient_accumulation_steps_dpo: int = 4
    beta_dpo: float = 0.1
    logging_steps_dpo: int = 50
    save_steps_dpo: int = 500


    # ---- DPO mining ----
    dpo_pair_strategy: str = 'hybrid'   # 'model_vs_model' | 'gt_anchor' | 'hybrid'
    dpo_target_pairs: int = 400
    dpo_max_processed_prompts: int = 800
    dpo_max_walltime_sec: Optional[int] = None
    dpo_num_samples_per_prompt: int = 3
    dpo_mining_max_new_tokens: int = 640
    dpo_mining_temperature: float = 0.80
    dpo_mining_top_p: float = 0.97
    dpo_mining_top_k: int = 50
    dpo_mining_include_greedy: bool = False
    dpo_min_margin: float = 0.06
    dpo_fast_score_size: int = 128
    dpo_log_every: int = 5
    dpo_save_every_pairs: int = 20
    dpo_score_workers: int = 2
    dpo_use_reranker_in_mining: bool = False
    dpo_resume: bool = True
    dpo_full_score_top_k: int = 2
    dpo_full_score_bottom_k: int = 1
    dpo_render_cache_items: int = 2048
    dpo_trace_every_logs: bool = True
    dpo_recent_window: int = 100
    dpo_rate_collapse_threshold: int = 6
    dpo_step_debug_first_n: int = 3
    dpo_step_debug_every: int = 25
    dpo_mining_use_dpo_adapter_if_present: bool = False

    # ---- DPO stability / debugging ----
    debug_cuda_launch_blocking: bool = False
    dpo_preflight_smoke_test: bool = True
    dpo_preflight_prompt: str = 'a simple black circle centered on a white background'
    dpo_preflight_max_new_tokens: int = 96
    dpo_abort_after_consecutive_generate_failures: int = 3
    dpo_abort_if_zero_pairs_after_processed: int = 25
    dpo_error_print_first_n: int = 3
    dpo_error_print_every: int = 25

    # ---- GT-anchor / hybrid DPO controls ----
    dpo_gt_anchor_fraction: float = 0.30
    dpo_gt_anchor_min_margin: float = 0.05
    dpo_gt_anchor_max_fraction_of_final_pairs: float = 0.35
    dpo_use_gt_as_chosen: bool = True
    dpo_gt_anchor_use_hard_negative: bool = True

    # ---- inference ----
    inference_num_candidates: int = 6
    inference_max_new_tokens: int = 1152
    inference_temperature: float = 0.72
    inference_top_p: float = 0.95
    inference_top_k: int = 50
    inference_repetition_penalty: float = 1.03

    # ---- reranking ----
    enable_clip_rerank: bool = False
    clip_model_dir: Optional[str] = None
    rerank_length_target: int = 2200

    # ---- saved dirs ----
    sft_adapter_dir: str = field(default_factory=lambda: f'{PROJECT_DIR}/outputs/sft_adapter')
    dpo_pairs_path: str = field(default_factory=lambda: f'{PROJECT_DIR}/outputs/dpo_pairs.parquet')
    dpo_adapter_dir: str = field(default_factory=lambda: f'{PROJECT_DIR}/outputs/dpo_adapter')
    submission_path: str = field(default_factory=lambda: f'{PROJECT_DIR}/submission_final.csv')
    dpo_trace_path: str = field(default_factory=lambda: f'{PROJECT_DIR}/outputs/dpo_mining_trace.jsonl')
    dpo_bad_samples_path: str = field(default_factory=lambda: f'{PROJECT_DIR}/outputs/dpo_bad_samples.jsonl')

CFG = Config()
os.makedirs(CFG.output_dir, exist_ok=True)

for path in [CFG.train_csv, CFG.test_csv]:
    print(f'{path}: {"FOUND" if os.path.exists(path) else "NOT FOUND ⚠️"}')
print('sft_adapter_dir:', CFG.sft_adapter_dir)
print('dpo_adapter_dir:', CFG.dpo_adapter_dir)


## 2. Utility Functions

In [ ]:
def bf16_supported() -> bool:
    return bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())

def safe_text(value: object) -> str:
    if value is None: return ''
    return str(value).strip()

def trim_float_text(text: str, digits: int = 2) -> str:
    def _fmt(match):
        x = float(match.group(0))
        out = f'{x:.{digits}f}'.rstrip('0').rstrip('.')
        return '0' if out == '-0' else out
    return FLOAT_REGEX.sub(_fmt, text)

def infer_color_from_prompt(prompt: str) -> str:
    low = prompt.lower()
    for c in COLOR_WORDS:
        if c in low: return 'gray' if c == 'grey' else c
    return 'gray'

def fallback_svg(prompt: str) -> str:
    fill = infer_color_from_prompt(prompt)
    low = prompt.lower()
    if 'triangle' in low:
        body = f'<polygon points="128,48 208,208 48,208" fill="{fill}"/>'
    elif 'square' in low or 'box' in low:
        body = f'<rect x="64" y="64" width="128" height="128" rx="18" fill="{fill}"/>'
    elif 'star' in low:
        body = f'<polygon fill="{fill}" points="128,40 148,96 208,96 160,132 178,192 128,154 78,192 96,132 48,96 108,96"/>'
    else:
        body = f'<circle cx="128" cy="128" r="72" fill="{fill}"/>'
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" '
        'viewBox="0 0 256 256">'
        '<rect width="256" height="256" fill="white"/>'
        f'{body}</svg>'
    )

print('Utilities loaded.')

## 3. SVG Recovery / Sanitization / Validation

In [ ]:
def extract_svg_candidates(raw_text: str) -> List[str]:
    raw_text = safe_text(raw_text)
    cands = [m.group(0).strip() for m in SVG_REGEX.finditer(raw_text)]
    if cands:
        return cands
    if '<svg' in raw_text.lower():
        start = raw_text.lower().find('<svg')
        partial = raw_text[start:].strip()
        if '</svg>' not in partial.lower():
            partial += '</svg>'
        cands.append(partial)
    return cands


def parse_xml_recover(svg_text: str) -> Optional[LET._Element]:
    if not svg_text:
        return None
    parser = LET.XMLParser(
        recover=True,
        resolve_entities=False,
        no_network=True,
        remove_comments=False,
        huge_tree=False,
    )
    try:
        return LET.fromstring(svg_text.encode('utf-8', errors='ignore'), parser=parser)
    except (LET.XMLSyntaxError, ValueError):
        return None


def local_tag(tag) -> str:
    if tag is None or not isinstance(tag, str):
        return ''
    return tag.rsplit('}', 1)[-1] if '}' in tag else tag


def drop_non_element_nodes(root: LET._Element) -> None:
    for elem in list(root.iter()):
        if not isinstance(getattr(elem, 'tag', None), str):
            parent = elem.getparent()
            if parent is not None:
                parent.remove(elem)


def is_external_ref(value: str) -> bool:
    value = safe_text(value).strip()
    if not value:
        return False
    low = value.lower()
    if low.startswith('#') or low.startswith('url(#'):
        return False
    if low.startswith('http:') or low.startswith('https:') or low.startswith('//'):
        return True
    if 'url(http' in low or 'url(https' in low:
        return True
    return False


def sanitize_attrs(elem: LET._Element) -> None:
    to_delete = []
    for k, v in list(elem.attrib.items()):
        lk = local_tag(k).lower()
        vv = safe_text(v)
        if lk.startswith(DANGEROUS_ATTR_PREFIXES):
            to_delete.append(k)
            continue
        if lk in REF_ATTRS and is_external_ref(vv):
            to_delete.append(k)
            continue
        if lk == 'style':
            low = vv.lower()
            if '@import' in low or 'url(http' in low or 'url(https' in low:
                to_delete.append(k)
                continue
    for k in to_delete:
        elem.attrib.pop(k, None)


def unwrap_or_drop_disallowed(root: LET._Element) -> None:
    for child in list(root):
        unwrap_or_drop_disallowed(child)
    for child in list(root):
        tag = local_tag(child.tag)
        if tag not in ALLOWED_TAGS:
            idx = root.index(child)
            for grand in list(child):
                root.insert(idx, grand)
                idx += 1
            root.remove(child)


def cleanup_svg_namespaces(root: LET._Element) -> None:
    for k in list(root.attrib.keys()):
        if local_tag(k).lower() == 'xmlns' or safe_text(k).lower().startswith('xmlns:'):
            root.attrib.pop(k, None)
    try:
        LET.cleanup_namespaces(root)
    except Exception:
        pass


def normalize_root(root: LET._Element) -> None:
    root.tag = f'{{{SVG_NS}}}svg'
    cleanup_svg_namespaces(root)
    root.set('width', '256')
    root.set('height', '256')
    root.set('viewBox', '0 0 256 256')


def count_paths(root: LET._Element) -> int:
    return sum(1 for elem in root.iter() if local_tag(elem.tag) == 'path')


def element_importance(elem: LET._Element) -> float:
    tag = local_tag(elem.tag)
    raw_len = len(LET.tostring(elem, encoding='unicode'))
    return PRIORITY_TAGS.get(tag, 0.5) * (1.0 + min(raw_len, 5000) / 5000.0)


def prune_paths(root: LET._Element, max_paths: int = MAX_PATHS) -> None:
    path_nodes = []
    for parent in root.iter():
        for child in list(parent):
            if local_tag(child.tag) == 'path':
                path_nodes.append((element_importance(child), parent, child))
    if len(path_nodes) <= max_paths:
        return
    path_nodes.sort(key=lambda x: x[0], reverse=True)
    keep = set(id(x[2]) for x in path_nodes[:max_paths])
    for _, parent, child in path_nodes[max_paths:]:
        if id(child) not in keep and child.getparent() is parent:
            parent.remove(child)


def strip_low_value_nodes(root: LET._Element) -> None:
    for parent in root.iter():
        for child in list(parent):
            tag = local_tag(child.tag)
            if tag in {'title', 'desc'}:
                parent.remove(child)
                continue
            if tag == 'style':
                text = safe_text(child.text).lower()
                if not text or '@import' in text or 'url(http' in text:
                    parent.remove(child)


def serialize_svg(root: LET._Element, digits: int = 2) -> str:
    cleanup_svg_namespaces(root)
    text = LET.tostring(root, encoding='unicode', pretty_print=False)
    text = text.replace('ns0:', '').replace(':ns0', '')
    text = re.sub(r'>\s+<', '><', text)
    text = trim_float_text(text, digits=digits)
    text = re.sub(r'\s+xmlns="[^"]+"', '', text)
    text = text.replace('<svg', f'<svg xmlns="{SVG_NS}"', 1)
    text = re.sub(r'(xmlns="[^"]+")(\s+)+', r'', text)
    return text


def prune_to_length(root: LET._Element, max_chars: int = MAX_SVG_CHARS) -> str:
    strip_low_value_nodes(root)
    text = serialize_svg(root, digits=2)
    if len(text) <= max_chars:
        return text
    text = serialize_svg(root, digits=1)
    if len(text) <= max_chars:
        return text
    removable = []
    for parent in root.iter():
        for child in list(parent):
            if local_tag(child.tag) == 'svg':
                continue
            removable.append((element_importance(child), parent, child))
    removable.sort(key=lambda x: x[0])
    for _, parent, child in removable:
        if child.getparent() is parent:
            parent.remove(child)
        text = serialize_svg(root, digits=1)
        if len(text) <= max_chars:
            return text
    return text


def strict_xml_validate(svg_text: str) -> bool:
    try:
        StdET.fromstring(svg_text.encode('utf-8', errors='ignore'))
        return True
    except Exception:
        return False


def render_svg(svg_text: str) -> Optional[np.ndarray]:
    try:
        import cairosvg
        png_bytes = cairosvg.svg2png(
            bytestring=svg_text.encode('utf-8'),
            output_width=256,
            output_height=256,
        )
        return np.asarray(Image.open(io.BytesIO(png_bytes)).convert('RGB'))
    except Exception:
        return None


def strict_validate_svg(svg_text: str, require_render: bool = False) -> bool:
    if not svg_text or len(svg_text) > MAX_SVG_CHARS:
        return False
    if not strict_xml_validate(svg_text):
        return False
    root = parse_xml_recover(svg_text)
    if root is None or local_tag(root.tag) != 'svg':
        return False
    pc = 0
    for elem in list(root.iter()):
        if not isinstance(getattr(elem, 'tag', None), str):
            return False
        tag = local_tag(elem.tag)
        if tag not in ALLOWED_TAGS:
            return False
        sanitize_attrs(elem)
        if tag == 'path':
            pc += 1
    if pc > MAX_PATHS:
        return False
    if require_render and render_svg(svg_text) is None:
        return False
    return True


def sanitize_svg(svg_text: str, require_render: bool = True) -> Optional[str]:
    root = parse_xml_recover(svg_text)
    if root is None:
        return None
    if local_tag(root.tag) != 'svg':
        wrapper = LET.Element(f'{{{SVG_NS}}}svg')
        wrapper.append(root)
        root = wrapper

    for _ in range(2):
        drop_non_element_nodes(root)
        unwrap_or_drop_disallowed(root)
        normalize_root(root)
        for elem in list(root.iter()):
            if not isinstance(getattr(elem, 'tag', None), str):
                parent = elem.getparent()
                if parent is not None:
                    parent.remove(elem)
                continue
            elem.tag = f'{{{SVG_NS}}}{local_tag(elem.tag)}'
            sanitize_attrs(elem)
        prune_paths(root, MAX_PATHS)
        svg_text = prune_to_length(root, MAX_SVG_CHARS)
        if not strict_xml_validate(svg_text):
            return None
        root = parse_xml_recover(svg_text)
        if root is None or local_tag(root.tag) != 'svg':
            return None

    svg_text = prune_to_length(root, MAX_SVG_CHARS)
    if not strict_validate_svg(svg_text, require_render=require_render):
        return None
    return svg_text

print('SVG sanitization loaded.')


## 4. Training Data Prep

In [ ]:
def normalize_training_svg(svg_text: str) -> Optional[str]:
    svg_text = safe_text(svg_text)
    root = parse_xml_recover(svg_text)
    if root is None or local_tag(root.tag) != 'svg': return None
    return sanitize_svg(svg_text, require_render=False)

def build_clean_dataset(train_csv, eval_fraction):
    df = pd.read_csv(train_csv)
    cleaned_rows = []
    reasons = Counter()
    for _, row in df.iterrows():
        prompt = safe_text(row.get('prompt'))
        svg = safe_text(row.get('svg'))
        if len(prompt) < 3: reasons['bad_prompt'] += 1; continue
        cleaned = normalize_training_svg(svg)
        if cleaned is None: reasons['bad_svg'] += 1; continue
        cleaned_rows.append({'prompt': prompt, 'svg': cleaned})
    random.shuffle(cleaned_rows)
    n_eval = max(100, int(len(cleaned_rows) * eval_fraction))
    eval_rows = cleaned_rows[:n_eval]
    train_rows = cleaned_rows[n_eval:]
    print(f'Clean train={len(train_rows)} eval={len(eval_rows)} rejected={sum(reasons.values())}')
    print(f'Reject reasons: {dict(reasons)}')
    return Dataset.from_list(train_rows), Dataset.from_list(eval_rows), pd.DataFrame(cleaned_rows)

def format_chat_example(prompt, svg, system_prompt):
    return (
        f'<|im_start|>system\n{system_prompt}<|im_end|>\n'
        f'<|im_start|>user\n{prompt}<|im_end|>\n'
        f'<|im_start|>assistant\n{svg}<|im_end|>'
    )

print('Data prep loaded.')

## 5. SFT Training

In [ ]:
def run_sft(config):
    from unsloth import FastLanguageModel
    from trl import SFTConfig, SFTTrainer

    train_ds, eval_ds, _ = build_clean_dataset(config.train_csv, config.eval_fraction)
    def _map(ex): return {'text': format_chat_example(ex['prompt'], ex['svg'], config.system_prompt)}
    train_fmt = train_ds.map(_map, remove_columns=train_ds.column_names)
    eval_fmt = eval_ds.map(_map, remove_columns=eval_ds.column_names)

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=config.model_name_train, max_seq_length=config.max_seq_length,
        dtype=None, load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model, r=config.lora_r, lora_alpha=config.lora_alpha,
        lora_dropout=config.lora_dropout, bias='none',
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        use_gradient_checkpointing='unsloth', random_state=SEED,
    )
    tokenizer = _normalize_tokenizer_for_text_generation(tokenizer)

    args = SFTConfig(
        output_dir=os.path.join(config.output_dir, 'sft_ckpts'),
        num_train_epochs=config.num_train_epochs_sft,
        per_device_train_batch_size=config.per_device_train_batch_size_sft,
        gradient_accumulation_steps=config.gradient_accumulation_steps_sft,
        learning_rate=config.learning_rate_sft,
        warmup_ratio=config.warmup_ratio_sft, weight_decay=config.weight_decay_sft,
        max_grad_norm=config.max_grad_norm_sft,
        fp16=not bf16_supported(), bf16=bf16_supported(),
        logging_steps=config.logging_steps, eval_strategy='steps', eval_steps=config.eval_steps,
        save_strategy='steps', save_steps=config.save_steps,
        save_total_limit=config.save_total_limit, load_best_model_at_end=False,
        report_to='none', optim='paged_adamw_8bit', lr_scheduler_type='cosine',
        seed=SEED, max_length=config.max_seq_length, packing=False, dataset_text_field='text',
    )
    trainer = SFTTrainer(model=model, processing_class=tokenizer,
                         train_dataset=train_fmt, eval_dataset=eval_fmt, args=args)
    trainer.train()
    os.makedirs(config.sft_adapter_dir, exist_ok=True)
    trainer.save_model(config.sft_adapter_dir)
    tokenizer.save_pretrained(config.sft_adapter_dir)
    print(f'[OK] SFT adapter saved -> {config.sft_adapter_dir}')

if RUN_SFT:
    run_sft(CFG)

## 6. Local Scorer for DPO

In [ ]:
def gray(arr: np.ndarray) -> np.ndarray:
    return np.dot(arr[..., :3], np.array([0.299, 0.587, 0.114])).astype(np.float32) / 255.0

def edge_f1(pred_rgb, gt_rgb):
    pred_edges = canny(gray(pred_rgb), sigma=1.0)
    gt_edges = canny(gray(gt_rgb), sigma=1.0)
    tp = np.logical_and(pred_edges, gt_edges).sum()
    fp = np.logical_and(pred_edges, ~gt_edges).sum()
    fn = np.logical_and(~pred_edges, gt_edges).sum()
    prec = tp / (tp + fp + 1e-8)
    rec = tp / (tp + fn + 1e-8)
    return float(2 * prec * rec / (prec + rec + 1e-8))

def tag_sequence(svg_text):
    root = parse_xml_recover(svg_text)
    return [local_tag(elem.tag) for elem in root.iter()] if root is not None else []

def tag_overlap_score(pred_svg, gt_svg):
    p, g = Counter(tag_sequence(pred_svg)), Counter(tag_sequence(gt_svg))
    if not p and not g: return 1.0
    common = sum((p & g).values())
    return float((2 * common) / max(sum(p.values()) + sum(g.values()), 1))

def compactness_score(pred_svg, gt_svg):
    return float(math.exp(-abs(math.log((len(pred_svg) + 50) / (len(gt_svg) + 50)))))

def proxy_competition_score(pred_svg, gt_svg):
    pred_svg = sanitize_svg(pred_svg, require_render=True)
    gt_svg = sanitize_svg(gt_svg, require_render=True)
    if pred_svg is None or gt_svg is None: return 0.0
    pred_im, gt_im = render_svg(pred_svg), render_svg(gt_svg)
    if pred_im is None or gt_im is None: return 0.0
    v_ssim = float(ssim(gray(pred_im), gray(gt_im), data_range=1.0))
    v_edge = edge_f1(pred_im, gt_im)
    visual = 0.7 * v_ssim + 0.3 * v_edge
    structure = tag_overlap_score(pred_svg, gt_svg)
    compact = compactness_score(pred_svg, gt_svg)
    return float((visual ** 0.90) * (structure ** 0.07) * (compact ** 0.03))

print('Scorer loaded.')

## 7. Inference Model & Generation

In [ ]:

ADAPTER_MODULE_CANDIDATES = (
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'up_proj', 'down_proj',
)
_ADAPTER_KEY_RE = re.compile(
    r'\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)\.lora_[AB](?:\.[^.]+)?\.weight$'
)
_LAYER_KEY_RE = re.compile(r'\.layers\.(\d+)\.')


def _model_leaf(name: Optional[str]) -> str:
    name = safe_text(name)
    return name.split('/')[-1].lower() if name else ''


def _listify_target_modules(value) -> List[str]:
    if value is None:
        return []
    if isinstance(value, (list, tuple, set)):
        return [str(x) for x in value]
    return [str(value)]


def _adapter_weight_file(adapter_dir: str) -> Optional[str]:
    if not adapter_dir or not os.path.exists(adapter_dir):
        return None
    for fname in ['adapter_model.safetensors', 'adapter_model.bin']:
        path = os.path.join(adapter_dir, fname)
        if os.path.exists(path):
            return path
    return None


def inspect_adapter_checkpoint(adapter_dir: str) -> Optional[dict]:
    weight_file = _adapter_weight_file(adapter_dir)
    if not weight_file:
        return None

    keys = []
    try:
        if weight_file.endswith('.safetensors'):
            from safetensors import safe_open
            with safe_open(weight_file, framework='pt', device='cpu') as f:
                keys = list(f.keys())
        else:
            state = torch.load(weight_file, map_location='cpu')
            if isinstance(state, dict) and 'state_dict' in state and isinstance(state['state_dict'], dict):
                state = state['state_dict']
            if isinstance(state, dict):
                keys = list(state.keys())
            else:
                raise RuntimeError('adapter_model.bin did not contain a dict-like state_dict')
    except Exception as e:
        print(f'[ADAPTER] failed to inspect adapter weights from {weight_file}: {e}')
        return None

    module_counts = Counter()
    layer_ids = set()
    matched_keys = 0
    sample_keys = []
    for key in keys:
        key = str(key)
        mm = _ADAPTER_KEY_RE.search(key)
        if not mm:
            continue
        matched_keys += 1
        module_counts[mm.group(1)] += 1
        lm = _LAYER_KEY_RE.search(key)
        if lm:
            layer_ids.add(int(lm.group(1)))
        if len(sample_keys) < 8:
            sample_keys.append(key)

    present_target_modules = sorted(module_counts.keys())
    return {
        'weight_file': weight_file,
        'num_state_keys': len(keys),
        'matched_lora_keys': matched_keys,
        'present_target_modules': present_target_modules,
        'module_counts': dict(sorted(module_counts.items())),
        'num_layers_with_lora': len(layer_ids),
        'min_layer': min(layer_ids) if layer_ids else None,
        'max_layer': max(layer_ids) if layer_ids else None,
        'sample_lora_keys': sample_keys,
    }


def read_adapter_signature(adapter_dir: str) -> Optional[dict]:
    if not adapter_dir or not os.path.exists(adapter_dir):
        return None
    try:
        from peft import PeftConfig
        cfg = PeftConfig.from_pretrained(adapter_dir)
        sig = {
            'adapter_dir': adapter_dir,
            'base_model_name_or_path': getattr(cfg, 'base_model_name_or_path', None),
            'peft_type': str(getattr(cfg, 'peft_type', None)),
            'task_type': str(getattr(cfg, 'task_type', None)),
            'r': getattr(cfg, 'r', None),
            'lora_alpha': getattr(cfg, 'lora_alpha', None),
            'lora_dropout': getattr(cfg, 'lora_dropout', None),
            'target_modules': sorted(_listify_target_modules(getattr(cfg, 'target_modules', None))),
        }
        ckpt_info = inspect_adapter_checkpoint(adapter_dir)
        if ckpt_info is not None:
            sig['checkpoint_present_target_modules'] = ckpt_info.get('present_target_modules', [])
            sig['checkpoint_module_counts'] = ckpt_info.get('module_counts', {})
            sig['checkpoint_num_layers_with_lora'] = ckpt_info.get('num_layers_with_lora')
            sig['checkpoint_sample_lora_keys'] = ckpt_info.get('sample_lora_keys', [])
        return sig
    except Exception as e:
        print(f'[ADAPTER] failed to read adapter config from {adapter_dir}: {e}')
        return None


def compare_adapter_to_notebook(config, sig: Optional[dict]) -> dict:
    if sig is None:
        return {'ok': True, 'mismatches': []}
    mismatches = []
    expected_targets = sorted(list(config.lora_target_modules))
    actual_targets = sorted(_listify_target_modules(sig.get('target_modules')))
    if expected_targets != actual_targets:
        mismatches.append({'field': 'target_modules', 'expected': expected_targets, 'actual': actual_targets})
    for field in ['r', 'lora_alpha', 'lora_dropout']:
        exp_v = getattr(config, field, None)
        act_v = sig.get(field)
        if exp_v is not None and act_v is not None and float(exp_v) != float(act_v):
            mismatches.append({'field': field, 'expected': exp_v, 'actual': act_v})
    ckpt_targets = sorted(_listify_target_modules(sig.get('checkpoint_present_target_modules')))
    if ckpt_targets and actual_targets and ckpt_targets != actual_targets:
        mismatches.append({'field': 'checkpoint_vs_config_target_modules', 'expected': actual_targets, 'actual': ckpt_targets})
    return {'ok': len(mismatches) == 0, 'mismatches': mismatches}


def print_adapter_signature(sig: Optional[dict], title: str = 'adapter'):
    if sig is None:
        print(f'[ADAPTER] {title}: <none>')
        return
    print(f'[ADAPTER] {title}:')
    for k in [
        'adapter_dir', 'base_model_name_or_path', 'peft_type', 'task_type',
        'r', 'lora_alpha', 'lora_dropout', 'target_modules',
        'checkpoint_present_target_modules', 'checkpoint_module_counts',
        'checkpoint_num_layers_with_lora', 'checkpoint_sample_lora_keys'
    ]:
        if k in sig:
            print(f'  - {k}: {sig.get(k)}')


def resolve_base_model_name(config, adapter_dir: Optional[str], purpose: str = 'infer') -> str:
    if config.adapter_base_model_override:
        override_name = safe_text(config.adapter_base_model_override)
        print(f'[ADAPTER] using explicit adapter_base_model_override -> {override_name}')
        return override_name

    sig = read_adapter_signature(adapter_dir) if adapter_dir else None
    adapter_base = safe_text(sig.get('base_model_name_or_path')) if sig else ''
    fallback = safe_text(config.model_name_train if purpose == 'train' else config.model_name_infer)

    if adapter_base:
        if fallback and adapter_base != fallback:
            print(
                f'[ADAPTER] {purpose} base mismatch: adapter_config wants {adapter_base} '
                f'but notebook config has {fallback}. Using adapter_config base exactly.'
            )
        return adapter_base
    return fallback


def _bnb_dtype():
    return torch.bfloat16 if bf16_supported() else torch.float16


def _short_error_text(exc: Exception, limit: int = 240) -> str:
    text = safe_text(repr(exc)).replace('\n', ' ').replace('\r', ' ')
    return text[:limit]


def _looks_like_adapter_key_warning(text: str) -> bool:
    text = safe_text(text).lower()
    return ('missing adapter keys' in text) or ('unexpected key' in text) or ('size mismatch' in text)


def _runtime_adapter_dir(config, adapter_dir: str, purpose: str = 'infer') -> str:
    if not adapter_dir or not os.path.exists(adapter_dir):
        return adapter_dir

    sig = read_adapter_signature(adapter_dir)
    ckpt = inspect_adapter_checkpoint(adapter_dir)

    if not ckpt:
        return adapter_dir

    cfg_targets = sorted(_listify_target_modules(sig.get('target_modules'))) if sig else []
    ckpt_targets = sorted(_listify_target_modules(ckpt.get('present_target_modules')))

    if not ckpt_targets:
        msg = (
            f'Could not infer any LoRA target modules from checkpoint weights in {adapter_dir}. '
            'This usually means the adapter checkpoint is incomplete or corrupted.'
        )
        if bool(config.adapter_repair_fail_if_no_checkpoint_modules):
            raise RuntimeError(msg)
        print(f'[ADAPTER-WARN] {msg}')
        return adapter_dir

    if cfg_targets == ckpt_targets:
        return adapter_dir

    if not bool(config.adapter_repair_target_modules_from_checkpoint):
        raise RuntimeError(
            'Saved adapter_config target_modules do not match the actual modules found in '
            f'adapter weights. adapter_config={cfg_targets}, checkpoint={ckpt_targets}'
        )

    if not set(ckpt_targets).issubset(set(ADAPTER_MODULE_CANDIDATES)):
        raise RuntimeError(
            f'Checkpoint target modules {ckpt_targets} contain unexpected names outside known candidates.'
        )

    os.makedirs(config.adapter_runtime_cache_dir, exist_ok=True)
    sig_payload = json.dumps({
        'adapter_dir': os.path.abspath(adapter_dir),
        'purpose': purpose,
        'cfg_targets': cfg_targets,
        'ckpt_targets': ckpt_targets,
        'base': sig.get('base_model_name_or_path') if sig else None,
    }, sort_keys=True)
    cache_key = hashlib.sha1(sig_payload.encode('utf-8')).hexdigest()[:12]
    patched_dir = os.path.join(
        config.adapter_runtime_cache_dir,
        f'{Path(adapter_dir).name}__{purpose}__{cache_key}'
    )
    os.makedirs(patched_dir, exist_ok=True)

    config_src = os.path.join(adapter_dir, 'adapter_config.json')
    config_dst = os.path.join(patched_dir, 'adapter_config.json')
    with open(config_src, 'r', encoding='utf-8') as f:
        cfg_json = json.load(f)
    cfg_json['target_modules'] = ckpt_targets
    with open(config_dst, 'w', encoding='utf-8') as f:
        json.dump(cfg_json, f, ensure_ascii=False, indent=2)

    for fname in ['adapter_model.safetensors', 'adapter_model.bin', 'tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json', 'chat_template.jinja']:
        src = os.path.join(adapter_dir, fname)
        if os.path.exists(src):
            dst = os.path.join(patched_dir, fname)
            if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
                shutil.copy2(src, dst)

    manifest = {
        'source_adapter_dir': os.path.abspath(adapter_dir),
        'patched_for_purpose': purpose,
        'original_target_modules': cfg_targets,
        'checkpoint_target_modules': ckpt_targets,
        'checkpoint_module_counts': ckpt.get('module_counts', {}),
        'base_model_name_or_path': sig.get('base_model_name_or_path') if sig else None,
    }
    with open(os.path.join(patched_dir, 'runtime_patch_manifest.json'), 'w', encoding='utf-8') as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

    print(
        '[ADAPTER-REPAIR] patched adapter_config target_modules based on checkpoint weights: '
        f'{cfg_targets} -> {ckpt_targets}'
    )
    print(f'[ADAPTER-REPAIR] runtime adapter dir -> {patched_dir}')
    return patched_dir


def _assert_adapter_signature(config, adapter_sig: Optional[dict], purpose: str):
    if adapter_sig is None:
        return
    cmp = compare_adapter_to_notebook(config, adapter_sig)
    if not cmp['ok']:
        print(f'[ADAPTER] notebook LoRA config differs from saved {purpose} adapter config.')
        for mm in cmp['mismatches']:
            print('  -', mm)
        print("[ADAPTER] Loader will prioritize the saved adapter checkpoint over notebook defaults.")


def _text_tokenizer(obj):
    if obj is None:
        return None
    for attr in ('tokenizer', 'text_tokenizer'):
        inner = getattr(obj, attr, None)
        if inner is not None and inner is not obj:
            return inner
    return obj


def _tok_get(obj, name, default=None):
    if obj is not None and hasattr(obj, name):
        value = getattr(obj, name)
        if value is not None:
            return value
    inner = _text_tokenizer(obj)
    if inner is not None and hasattr(inner, name):
        value = getattr(inner, name)
        if value is not None:
            return value
    return default


def _tok_set(obj, name, value):
    done = False
    if obj is not None and hasattr(obj, name):
        try:
            setattr(obj, name, value)
            done = True
        except Exception:
            pass
    inner = _text_tokenizer(obj)
    if inner is not None and inner is not obj and hasattr(inner, name):
        try:
            setattr(inner, name, value)
            done = True
        except Exception:
            pass
    return done


def _normalize_tokenizer_for_text_generation(tokenizer):
    _tok_set(tokenizer, 'padding_side', 'left')
    pad_token = _tok_get(tokenizer, 'pad_token')
    eos_token = _tok_get(tokenizer, 'eos_token')
    eos_token_id = _tok_get(tokenizer, 'eos_token_id')
    if pad_token is None and eos_token is not None:
        _tok_set(tokenizer, 'pad_token', eos_token)
        pad_token = eos_token
    if _tok_get(tokenizer, 'pad_token_id') is None and eos_token_id is not None:
        _tok_set(tokenizer, 'pad_token_id', eos_token_id)
    return tokenizer


def _tok_apply_chat_template(tokenizer, messages, tokenize=False, add_generation_prompt=True):
    if hasattr(tokenizer, 'apply_chat_template'):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=tokenize,
            add_generation_prompt=add_generation_prompt,
        )
    inner = _text_tokenizer(tokenizer)
    if inner is not None and hasattr(inner, 'apply_chat_template'):
        return inner.apply_chat_template(
            messages,
            tokenize=tokenize,
            add_generation_prompt=add_generation_prompt,
        )
    raise AttributeError('Tokenizer/processor does not expose apply_chat_template')


def _tok_decode(tokenizer, output_ids, skip_special_tokens=False):
    if hasattr(tokenizer, 'decode'):
        return tokenizer.decode(output_ids, skip_special_tokens=skip_special_tokens)
    inner = _text_tokenizer(tokenizer)
    if inner is not None and hasattr(inner, 'decode'):
        return inner.decode(output_ids, skip_special_tokens=skip_special_tokens)
    raise AttributeError('Tokenizer/processor does not expose decode')


def _tok_pad_token_id(tokenizer):
    return _tok_get(tokenizer, 'pad_token_id')


def _tok_eos_token_id(tokenizer):
    return _tok_get(tokenizer, 'eos_token_id')


def _load_unsloth_model_from_name(model_name_or_path: str, max_seq_length: int, for_inference: bool):
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name_or_path,
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=True,
    )
    tokenizer = _normalize_tokenizer_for_text_generation(tokenizer)
    if for_inference:
        FastLanguageModel.for_inference(model)
        model.eval()
    return model, tokenizer


def _try_unsloth_direct_adapter_load(config, adapter_dir: str, purpose: str, allow_runtime_patch: bool = True):
    attempts = []
    direct_first = bool(getattr(config, 'adapter_direct_load_first', True))
    runtime_dir = _runtime_adapter_dir(config, adapter_dir, purpose=purpose) if allow_runtime_patch else adapter_dir
    ordered = []
    if direct_first:
        ordered.append(('adapter_dir', adapter_dir))
        if runtime_dir and os.path.abspath(runtime_dir) != os.path.abspath(adapter_dir):
            ordered.append(('runtime_patched_adapter_dir', runtime_dir))
    else:
        ordered.append(('runtime_patched_adapter_dir', runtime_dir))
        if runtime_dir and os.path.abspath(runtime_dir) != os.path.abspath(adapter_dir):
            ordered.append(('adapter_dir', adapter_dir))

    for label, path in ordered:
        if not path or not os.path.exists(path):
            continue
        try:
            print(f'[LOAD] {purpose} via Unsloth direct adapter path -> {path} ({label})')
            model, tokenizer = _load_unsloth_model_from_name(
                path,
                max_seq_length=config.max_seq_length,
                for_inference=(purpose == 'infer'),
            )
            print(f'[LOAD] {purpose} direct adapter load succeeded from {path}')
            return model, tokenizer, path
        except Exception as e:
            err = _short_error_text(e, limit=800)
            attempts.append({'path': path, 'label': label, 'error': err})
            print(f'[LOAD-WARN] {purpose} direct adapter load failed from {path}: {err}')

    lines = [f"- {a['label']}: {a['path']} :: {a['error']}" for a in attempts]
    msg = 'Unsloth direct adapter load failed. Attempts:\n' + '\n'.join(lines)
    raise RuntimeError(msg)


def load_text_inference_model(config, adapter_dir):
    runtime_adapter_dir = _runtime_adapter_dir(config, adapter_dir, purpose='infer') if adapter_dir else adapter_dir
    adapter_sig = read_adapter_signature(runtime_adapter_dir)
    print_adapter_signature(adapter_sig, title='generation adapter')
    _assert_adapter_signature(config, adapter_sig, purpose='generation')

    if adapter_dir and os.path.exists(adapter_dir):
        model, tokenizer, loaded_from = _try_unsloth_direct_adapter_load(
            config,
            adapter_dir=adapter_dir,
            purpose='infer',
            allow_runtime_patch=True,
        )
        print(f'[LOAD] generation model source -> {loaded_from}')
        return model, tokenizer

    base_model_name = resolve_base_model_name(config, adapter_dir=runtime_adapter_dir, purpose='infer')
    print(f'[LOAD] generation base model -> {base_model_name}')
    model, tokenizer = _load_unsloth_model_from_name(
        base_model_name,
        max_seq_length=config.max_seq_length,
        for_inference=True,
    )
    return model, tokenizer


def load_trainable_model_from_sft_adapter(config, sft_adapter_dir):
    runtime_adapter_dir = _runtime_adapter_dir(config, sft_adapter_dir, purpose='train') if sft_adapter_dir else sft_adapter_dir
    adapter_sig = read_adapter_signature(runtime_adapter_dir)
    print_adapter_signature(adapter_sig, title='trainable SFT adapter')
    _assert_adapter_signature(config, adapter_sig, purpose='trainable SFT')

    if sft_adapter_dir and os.path.exists(sft_adapter_dir):
        model, tokenizer, loaded_from = _try_unsloth_direct_adapter_load(
            config,
            adapter_dir=sft_adapter_dir,
            purpose='train',
            allow_runtime_patch=True,
        )
        print(f'[LOAD] trainable model source -> {loaded_from}')
        return model, tokenizer

    from unsloth import FastLanguageModel
    base_model_name = resolve_base_model_name(config, adapter_dir=runtime_adapter_dir, purpose='train')
    print(f'[LOAD] train base model -> {base_model_name}')
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=base_model_name,
        max_seq_length=config.max_seq_length,
        dtype=None,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=config.lora_r,
        lora_alpha=config.lora_alpha,
        lora_dropout=config.lora_dropout,
        bias='none',
        target_modules=list(config.lora_target_modules),
        use_gradient_checkpointing='unsloth',
        random_state=SEED,
    )
    tokenizer = _normalize_tokenizer_for_text_generation(tokenizer)
    return model, tokenizer


def build_chat_prompt(tokenizer, system_prompt, prompt):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': prompt},
    ]
    return _tok_apply_chat_template(tokenizer, messages, tokenize=False, add_generation_prompt=True)


def decode_assistant_text(tokenizer, output_ids):
    decoded = _tok_decode(tokenizer, output_ids, skip_special_tokens=False)
    if '<|im_start|>assistant\n' in decoded:
        decoded = decoded.split('<|im_start|>assistant\n', 1)[-1]
    if '</think>' in decoded:
        decoded = decoded.split('</think>', 1)[-1]
    for tok in ['<|im_end|>', '<|im_start|>', '<|endoftext|>']:
        if tok in decoded:
            decoded = decoded.split(tok, 1)[0]
    return decoded.strip()


print('Inference / adapter loading functions loaded.')


In [ ]:
class LightReranker:
    def __init__(self, config):
        self.config = config
        self.clip_model = None
        self.clip_processor = None
        if config.enable_clip_rerank and config.clip_model_dir and os.path.exists(config.clip_model_dir):
            try:
                from transformers import CLIPModel, CLIPProcessor
                self.clip_model = CLIPModel.from_pretrained(config.clip_model_dir).eval()
                self.clip_processor = CLIPProcessor.from_pretrained(config.clip_model_dir)
                if torch.cuda.is_available(): self.clip_model = self.clip_model.to('cuda')
                print('[OK] CLIP reranker loaded')
            except Exception as e: print(f'[WARN] CLIP reranker failed: {e}')

    def clip_score(self, prompt, image_rgb):
        if self.clip_model is None: return 0.0
        pil = Image.fromarray(image_rgb)
        inputs = self.clip_processor(text=[prompt], images=[pil], return_tensors='pt', padding=True)
        if torch.cuda.is_available(): inputs = {k: v.to('cuda') for k, v in inputs.items()}
        with torch.no_grad():
            out = self.clip_model(**inputs)
            img = out.image_embeds / out.image_embeds.norm(dim=-1, keepdim=True)
            txt = out.text_embeds / out.text_embeds.norm(dim=-1, keepdim=True)
            return float((img * txt).sum().item())

    def heuristic_score(self, prompt, svg_text, image_rgb):
        score = 0.0
        low_prompt, low_svg = prompt.lower(), svg_text.lower()
        for c in COLOR_WORDS:
            if c in low_prompt and c in low_svg: score += 0.08
        target = self.config.rerank_length_target
        score += 0.25 * math.exp(-abs(math.log((len(svg_text) + 50) / (target + 50))))
        root = parse_xml_recover(svg_text)
        path_pen = count_paths(root) / MAX_PATHS if root is not None else 1.0
        score += 0.10 * (1.0 - min(path_pen, 1.0))
        std = float(gray(image_rgb).std())
        score += 0.15 * min(std / 0.20, 1.0)
        return score

    def score(self, prompt, svg_text):
        img = render_svg(svg_text)
        if img is None: return -1e9
        score = self.heuristic_score(prompt, svg_text, img)
        if self.clip_model is not None: score += 2.0 * self.clip_score(prompt, img)
        return float(score)

print('Reranker loaded.')

In [ ]:
def _model_input_device(model):
    try:
        return next(model.parameters()).device
    except Exception:
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def _prepare_generate_inputs(tokenizer, text, model):
    device = _model_input_device(model)
    errors = []

    attempts = [
        lambda: tokenizer(text=text, images=None, videos=None, return_tensors='pt'),
        lambda: tokenizer(text=[text], images=None, videos=None, return_tensors='pt'),
        lambda: tokenizer(text=text, return_tensors='pt'),
        lambda: tokenizer(text=[text], return_tensors='pt'),
    ]

    inner = _text_tokenizer(tokenizer)
    if inner is not None and inner is not tokenizer:
        attempts.extend([
            lambda: inner(text, return_tensors='pt'),
            lambda: inner([text], return_tensors='pt'),
        ])

    inputs = None
    for fn in attempts:
        try:
            inputs = fn()
            break
        except TypeError as e:
            errors.append(f'{type(e).__name__}: {e}')
        except ValueError as e:
            errors.append(f'{type(e).__name__}: {e}')
        except Exception as e:
            errors.append(f'{type(e).__name__}: {e}')

    if inputs is None:
        preview = text[:300].replace('\n', '\\n')
        raise RuntimeError(
            'Failed to tokenize text-only generation prompt. '
            f'Prompt preview={preview!r}. Attempts={errors[:6]}'
        )

    moved = {}
    for k, v in inputs.items():
        moved[k] = v.to(device) if hasattr(v, 'to') else v
    return moved


def generation_smoke_test(model, tokenizer, config, prompt: Optional[str] = None, max_new_tokens: Optional[int] = None):
    prompt = safe_text(prompt) or safe_text(config.dpo_preflight_prompt)
    max_new_tokens = int(max_new_tokens or config.dpo_preflight_max_new_tokens)
    text = build_chat_prompt(tokenizer, config.system_prompt, prompt)
    inputs = _prepare_generate_inputs(tokenizer, text, model)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.0,
            pad_token_id=_tok_pad_token_id(tokenizer),
            eos_token_id=_tok_eos_token_id(tokenizer),
            use_cache=True,
        )
    snippet = decode_assistant_text(tokenizer, out_ids[0])[:200]
    print(f'[SMOKE] generation ok | prompt_chars={len(prompt)} | snippet={snippet!r}')
    return snippet


def generate_candidates(model, tokenizer, config, prompt, num_candidates,
                         temperature, top_p, top_k,
                         max_new_tokens: Optional[int] = None,
                         include_greedy: bool = True,
                         require_render: bool = True):
    text = build_chat_prompt(tokenizer, config.system_prompt, prompt)
    inputs = _prepare_generate_inputs(tokenizer, text, model)
    candidates = []
    max_new_tokens = int(max_new_tokens or config.inference_max_new_tokens)

    def _collect_from_output_ids(output_ids_batch):
        if output_ids_batch.ndim == 1:
            output_ids_batch = output_ids_batch.unsqueeze(0)
        for out_ids in output_ids_batch:
            assistant = decode_assistant_text(tokenizer, out_ids)
            raw_svgs = extract_svg_candidates(assistant)
            if not raw_svgs and '<svg' in assistant.lower():
                raw_svgs = [assistant]
            for raw in raw_svgs:
                repaired = sanitize_svg(raw, require_render=require_render)
                if repaired is not None:
                    candidates.append(repaired)
                    break

    with torch.no_grad():
        try:
            sampled_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                num_return_sequences=max(1, int(num_candidates)),
                temperature=temperature,
                top_p=top_p,
                top_k=top_k,
                repetition_penalty=config.inference_repetition_penalty,
                pad_token_id=_tok_pad_token_id(tokenizer),
                eos_token_id=_tok_eos_token_id(tokenizer),
                use_cache=True,
            )
            _collect_from_output_ids(sampled_ids)
        except RuntimeError as e:
            msg = safe_text(repr(e)).lower()
            if 'out of memory' in msg:
                if torch.cuda.is_available():
                    try:
                        torch.cuda.empty_cache()
                    except Exception:
                        pass
                try:
                    for _ in range(max(1, int(num_candidates))):
                        out_ids = model.generate(
                            **inputs,
                            max_new_tokens=max_new_tokens,
                            do_sample=True,
                            temperature=temperature,
                            top_p=top_p,
                            top_k=top_k,
                            repetition_penalty=config.inference_repetition_penalty,
                            pad_token_id=_tok_pad_token_id(tokenizer),
                            eos_token_id=_tok_eos_token_id(tokenizer),
                            use_cache=True,
                        )[0]
                        _collect_from_output_ids(out_ids)
                except Exception as e2:
                    raise RuntimeError(f'generate_candidates OOM fallback failed: {repr(e2)}') from e2
            else:
                raise RuntimeError(f'generate_candidates sampled generation failed: {repr(e)}') from e
        except Exception as e:
            raise RuntimeError(f'generate_candidates sampled generation failed: {repr(e)}') from e

        if include_greedy:
            try:
                out_ids = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    repetition_penalty=1.01,
                    pad_token_id=_tok_pad_token_id(tokenizer),
                    eos_token_id=_tok_eos_token_id(tokenizer),
                    use_cache=True,
                )[0]
                _collect_from_output_ids(out_ids)
            except Exception as e:
                raise RuntimeError(f'generate_candidates greedy generation failed: {repr(e)}') from e

    return list(dict.fromkeys(candidates))  # unique, order-preserving


def choose_best_candidate(reranker, prompt, candidates):
    if not candidates:
        return None
    scored = [(reranker.score(prompt, c), c) for c in candidates]
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][1]


def generate_svg_final(model, tokenizer, reranker, config, prompt):
    cands = generate_candidates(
        model, tokenizer, config, prompt,
        num_candidates=config.inference_num_candidates,
        temperature=config.inference_temperature,
        top_p=config.inference_top_p, top_k=config.inference_top_k,
        max_new_tokens=config.inference_max_new_tokens,
        include_greedy=True,
        require_render=True,
    )
    best = choose_best_candidate(reranker, prompt, cands)
    if best is not None:
        return best
    fb = fallback_svg(prompt)
    fb = sanitize_svg(fb, require_render=True)
    return fb if fb is not None else fallback_svg(prompt)


print('Generation functions loaded.')


## 8. DPO Pair Mining (Optimized)

This section replaces the original `mine_dpo_pairs` with a Colab-stable implementation designed for:
- shorter mining-time generation
- candidate dedup + fast sanitize
- GT render caching
- two-stage scoring
- optional **hybrid GT-anchor** preference mining
- structured progress logging and checkpointing
- resume after interruption


In [ ]:
def _now():
    return time.monotonic()


def _sha1(s: str) -> str:
    return hashlib.sha1(s.encode('utf-8', errors='ignore')).hexdigest()


def _safe_subprocess(cmd):
    try:
        return subprocess.check_output(cmd, stderr=subprocess.STDOUT).decode('utf-8', errors='ignore').strip()
    except Exception:
        return None


def _gpu_stats():
    out = _safe_subprocess([
        'nvidia-smi',
        '--query-gpu=utilization.gpu,utilization.memory,memory.used,memory.total',
        '--format=csv,noheader,nounits',
    ])
    if not out:
        return None
    try:
        util_gpu, util_mem, mem_used, mem_total = [x.strip() for x in out.split(',')]
        return {
            'gpu_util_pct': float(util_gpu),
            'mem_util_pct': float(util_mem),
            'mem_used_mb': float(mem_used),
            'mem_total_mb': float(mem_total),
        }
    except Exception:
        return None


class _LRU:
    def __init__(self, max_items=2048):
        self.max_items = int(max_items)
        self.data = OrderedDict()

    def get(self, key):
        if key not in self.data:
            return None
        value = self.data.pop(key)
        self.data[key] = value
        return value

    def put(self, key, value):
        if self.max_items <= 0:
            return
        if key in self.data:
            self.data.pop(key)
        self.data[key] = value
        while len(self.data) > self.max_items:
            self.data.popitem(last=False)


def _render_svg_gray_u8(svg: str, size: int) -> np.ndarray:
    import cairosvg
    png_bytes = cairosvg.svg2png(
        bytestring=svg.encode('utf-8', errors='ignore'),
        output_width=size,
        output_height=size,
    )
    img = Image.open(io.BytesIO(png_bytes)).convert('L')
    arr = np.array(img, dtype=np.uint8)
    if arr.shape != (size, size):
        arr = np.array(img.resize((size, size), resample=Image.BILINEAR), dtype=np.uint8)
    return arr


def _edge_f1_bool(pred_edge: np.ndarray, gt_edge: np.ndarray) -> float:
    pred = pred_edge.astype(bool)
    gt = gt_edge.astype(bool)
    tp = np.logical_and(pred, gt).sum()
    fp = np.logical_and(pred, np.logical_not(gt)).sum()
    fn = np.logical_and(np.logical_not(pred), gt).sum()
    denom_p = tp + fp
    denom_r = tp + fn
    if denom_p == 0 or denom_r == 0:
        return 0.0
    prec = tp / denom_p
    rec = tp / denom_r
    denom = prec + rec
    return float(2 * prec * rec / denom) if denom > 0 else 0.0


def _write_jsonl(path: str, obj: dict):
    if not path:
        return
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(obj, ensure_ascii=False) + '\n')


def _safe_sanitize_candidate(svg_text: str, require_render: bool = False) -> Optional[str]:
    try:
        return sanitize_svg(svg_text, require_render=require_render)
    except Exception:
        return None


def _sanitize_gt(svg_text: str) -> Optional[str]:
    try:
        return sanitize_svg(svg_text, require_render=True)
    except Exception:
        return None


def _looks_like_cuda_side_error(exc: Exception) -> bool:
    text = safe_text(repr(exc)).lower()
    needles = [
        'device-side assert', 'cuda error', 'cublas', 'cusparse', 'illegal memory access',
        'misaligned address', 'an illegal memory access was encountered'
    ]
    return any(x in text for x in needles)


def _safe_empty_cache(cuda_poisoned: bool = False):
    if not torch.cuda.is_available():
        return
    if cuda_poisoned:
        print('[CUDA] skipping torch.cuda.empty_cache() because CUDA context is likely poisoned. Restart runtime before rerun.')
        return
    try:
        torch.cuda.empty_cache()
    except Exception as e:
        print(f'[CUDA] empty_cache skipped after exception: {repr(e)}')


def mine_dpo_pairs_optimized(config, adapter_dir):
    from concurrent.futures import ThreadPoolExecutor, as_completed

    t_start = _now()
    last_gpu_poll = 0.0
    cuda_poisoned = False

    if bool(config.debug_cuda_launch_blocking):
        os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
        print('[DEBUG] CUDA_LAUNCH_BLOCKING=1 enabled for this run.')

    print(f'[DPO-MINE] starting with adapter_dir={adapter_dir}')

    source_df = pd.read_csv(config.train_csv)
    source_df = source_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    source_df = source_df.head(int(config.dpo_max_processed_prompts)).reset_index(drop=True)

    rows = []
    done_prompt_hash = set()
    if config.dpo_resume and os.path.exists(config.dpo_pairs_path):
        try:
            prev = pd.read_parquet(config.dpo_pairs_path)
            rows = prev.to_dict('records')
            for r in rows:
                done_prompt_hash.add(_sha1(str(r.get('prompt', ''))[:2000]))
            print(f'[RESUME] loaded {len(rows)} existing pairs from {config.dpo_pairs_path}')
        except Exception as e:
            print(f'[RESUME] failed to load existing pairs: {e}')

    built = len(rows)
    gt_anchor_built = int(sum(1 for r in rows if str(r.get('pair_type', '')) == 'gt_anchor'))
    model_model_built = built - gt_anchor_built

    model, tokenizer = load_text_inference_model(config, adapter_dir=adapter_dir)
    print('[DPO-MINE] model + tokenizer loaded, entering mining loop')

    if bool(config.dpo_preflight_smoke_test):
        try:
            generation_smoke_test(
                model,
                tokenizer,
                config,
                prompt=safe_text(config.dpo_preflight_prompt),
                max_new_tokens=int(config.dpo_preflight_max_new_tokens),
            )
        except Exception as e:
            cuda_poisoned = cuda_poisoned or _looks_like_cuda_side_error(e)
            _write_jsonl(config.dpo_bad_samples_path, {
                'kind': 'preflight_smoke_test_exception',
                'stage': 'preflight_smoke_test',
                'error': repr(e),
                'traceback': traceback.format_exc(limit=6),
                'adapter_dir': adapter_dir,
            })
            raise RuntimeError(
                'Preflight generation smoke test failed before DPO mining. '
                f'Last error: {_short_error_text(e, limit=600)}'
            ) from e

    reranker = LightReranker(config) if config.dpo_use_reranker_in_mining else None

    gt_feat_cache = _LRU(max_items=1024)
    render_cache = _LRU(max_items=int(config.dpo_render_cache_items))
    score_cache = _LRU(max_items=4096)
    gt_self_score_cache = _LRU(max_items=1024)

    skip = defaultdict(int)
    recent_built = deque(maxlen=int(config.dpo_recent_window))
    gen_time_sum = 0.0
    score_time_sum = 0.0
    processed = 0
    consecutive_generate_failures = 0

    def _get_gt_feats(gt_svg: str):
        key = _sha1(gt_svg)
        cached = gt_feat_cache.get(key)
        if cached is not None:
            return cached
        gt_img = _render_svg_gray_u8(gt_svg, int(config.dpo_fast_score_size))
        gt_edge = canny(gt_img.astype(np.float32) / 255.0)
        feats = {'gt_img': gt_img, 'gt_edge': gt_edge, 'gt_len': len(gt_svg)}
        gt_feat_cache.put(key, feats)
        return feats

    def _fast_proxy_score(pred_svg: str, gt_svg: str, gt_feats: dict) -> float:
        pred_key = _sha1(pred_svg)
        gt_key = _sha1(gt_svg)
        skey = pred_key + ':' + gt_key + f':{config.dpo_fast_score_size}'
        cached = score_cache.get(skey)
        if cached is not None:
            return cached

        rkey = pred_key + f':{config.dpo_fast_score_size}'
        pred_img = render_cache.get(rkey)
        if pred_img is None:
            pred_img = _render_svg_gray_u8(pred_svg, int(config.dpo_fast_score_size))
            render_cache.put(rkey, pred_img)

        gt_img = gt_feats['gt_img']
        gt_edge = gt_feats['gt_edge']
        pred_edge = canny(pred_img.astype(np.float32) / 255.0)
        try:
            ssim_v = float(ssim(pred_img, gt_img, data_range=255))
        except Exception:
            ssim_v = 0.0
        edge_v = _edge_f1_bool(pred_edge, gt_edge)
        len_ratio = min(len(pred_svg), gt_feats['gt_len']) / max(max(len(pred_svg), gt_feats['gt_len']), 1)
        score = 0.72 * ssim_v + 0.25 * edge_v + 0.03 * float(len_ratio)
        score_cache.put(skey, score)
        return score

    def _get_gt_self_score(gt_svg: str) -> float:
        key = _sha1(gt_svg)
        cached = gt_self_score_cache.get(key)
        if cached is not None:
            return cached
        val = float(proxy_competition_score(gt_svg, gt_svg))
        gt_self_score_cache.put(key, val)
        return val

    def _score_candidates_two_stage(uniq: List[str], gt_svg: str):
        gt_feats = _get_gt_feats(gt_svg)
        fast_scores = []

        t0 = _now()
        if int(config.dpo_score_workers) > 1 and len(uniq) > 1:
            with ThreadPoolExecutor(max_workers=int(config.dpo_score_workers)) as ex:
                futs = {ex.submit(_fast_proxy_score, u, gt_svg, gt_feats): u for u in uniq}
                for fut in as_completed(futs):
                    u = futs[fut]
                    try:
                        s = float(fut.result())
                    except Exception:
                        s = -1.0
                    fast_scores.append((s, u))
        else:
            for u in uniq:
                try:
                    s = float(_fast_proxy_score(u, gt_svg, gt_feats))
                except Exception:
                    s = -1.0
                fast_scores.append((s, u))

        fast_scores.sort(key=lambda x: x[0], reverse=True)

        if len(uniq) <= 4:
            need_full = [u for _, u in fast_scores]
        else:
            top_part = [u for _, u in fast_scores[: int(config.dpo_full_score_top_k)]]
            bottom_part = [u for _, u in fast_scores[-int(config.dpo_full_score_bottom_k):]]
            need_full = list(dict.fromkeys(top_part + bottom_part))

        full_scores = []
        for u in need_full:
            try:
                full_scores.append((float(proxy_competition_score(u, gt_svg)), u))
            except Exception as e:
                skip['full_score_exception'] += 1
                _write_jsonl(config.dpo_bad_samples_path, {
                    'kind': 'full_score_exception',
                    'candidate_prefix': safe_text(u)[:400],
                    'gt_prefix': safe_text(gt_svg)[:400],
                    'error': repr(e),
                })

        full_scores.sort(key=lambda x: x[0], reverse=True)
        dt = _now() - t0
        return fast_scores, full_scores, dt

    def _choose_model_vs_model_pair(full_scores):
        if len(full_scores) < 2:
            skip['not_enough_full_scores_model_vs_model'] += 1
            return None
        best_score, best_svg = full_scores[0]
        worst_score, worst_svg = full_scores[-1]
        if (best_score - worst_score) < float(config.dpo_min_margin):
            skip['margin_too_small_model_vs_model'] += 1
            return None
        return {
            'pair_type': 'model_vs_model',
            'chosen': best_svg,
            'rejected': worst_svg,
            'chosen_score': float(best_score),
            'rejected_score': float(worst_score),
        }

    def _gt_anchor_allowed(current_built, current_gt_anchor_built):
        if not bool(config.dpo_use_gt_as_chosen):
            return False
        if str(config.dpo_pair_strategy) == 'model_vs_model':
            return False
        if current_built <= 0:
            return True
        frac = current_gt_anchor_built / max(current_built, 1)
        return frac < float(config.dpo_gt_anchor_max_fraction_of_final_pairs)

    def _choose_gt_anchor_pair(gt_svg, full_scores):
        gt_self = _get_gt_self_score(gt_svg)
        eligible = []
        for score_v, cand in full_scores:
            margin = gt_self - float(score_v)
            if margin >= float(config.dpo_gt_anchor_min_margin):
                eligible.append((float(score_v), cand, float(margin)))
        if not eligible:
            skip['gt_anchor_no_eligible_negative'] += 1
            return None

        eligible.sort(key=lambda x: x[0], reverse=True)
        if bool(config.dpo_gt_anchor_use_hard_negative):
            rej_score, rejected_svg, margin = eligible[0]
        else:
            rej_score, rejected_svg, margin = eligible[-1]

        return {
            'pair_type': 'gt_anchor',
            'chosen': gt_svg,
            'rejected': rejected_svg,
            'chosen_score': float(gt_self),
            'rejected_score': float(rej_score),
            'gt_margin': float(margin),
        }

    def _select_pair(pair_strategy, gt_svg, full_scores, current_built, current_gt_anchor_built):
        pair = None
        try_gt = _gt_anchor_allowed(current_built, current_gt_anchor_built) and pair_strategy in {'gt_anchor', 'hybrid'}
        if pair_strategy == 'gt_anchor':
            try_gt = True

        if pair_strategy == 'hybrid':
            use_gt_now = try_gt and (random.random() < float(config.dpo_gt_anchor_fraction))
        else:
            use_gt_now = try_gt and pair_strategy == 'gt_anchor'

        if use_gt_now:
            pair = _choose_gt_anchor_pair(gt_svg, full_scores)
            if pair is None and pair_strategy == 'hybrid':
                pair = _choose_model_vs_model_pair(full_scores)
        else:
            pair = _choose_model_vs_model_pair(full_scores)
            if pair is None and try_gt and pair_strategy == 'hybrid':
                pair = _choose_gt_anchor_pair(gt_svg, full_scores)
        return pair

    for i, row in source_df.iterrows():
        if built >= int(config.dpo_target_pairs):
            print(f'[STOP] reached target_pairs={config.dpo_target_pairs}')
            break
        if config.dpo_max_walltime_sec is not None:
            elapsed = _now() - t_start
            if elapsed >= float(config.dpo_max_walltime_sec):
                print(f'[STOP] reached dpo_max_walltime_sec={config.dpo_max_walltime_sec}')
                break

        processed += 1
        recent_built.append(0)
        prompt = safe_text(row.get('prompt', ''))
        gt_svg = normalize_training_svg(safe_text(row.get('svg', '')))
        loop_ok = False
        last_fail = None
        last_error_short = None
        stage = 'start'
        candidate_count = 0
        full_count = 0

        if processed <= int(config.dpo_step_debug_first_n) or (processed % int(config.dpo_step_debug_every) == 0):
            print(f'[DPO-STEP] start processed={processed} raw_index={i} prompt_chars={len(prompt)}')

        try:
            if not prompt:
                skip['empty_prompt'] += 1
                last_fail = 'empty_prompt'
                continue
            if gt_svg is None:
                skip['gt_svg_none'] += 1
                last_fail = 'gt_svg_none'
                continue

            stage = 'sanitize_gt'
            gt_svg = _sanitize_gt(gt_svg)
            if gt_svg is None:
                skip['gt_svg_invalid_after_sanitize'] += 1
                last_fail = 'gt_svg_invalid_after_sanitize'
                continue

            stage = 'prompt_template'
            prompt_text = build_chat_prompt(tokenizer, config.system_prompt, prompt)
            prompt_key = _sha1(prompt_text[:2000])
            if config.dpo_resume and prompt_key in done_prompt_hash:
                skip['resume_skip'] += 1
                last_fail = 'resume_skip'
                continue

            stage = 'generate'
            t_gen0 = _now()
            cands = generate_candidates(
                model, tokenizer, config, prompt,
                num_candidates=int(config.dpo_num_samples_per_prompt),
                temperature=float(config.dpo_mining_temperature),
                top_p=float(config.dpo_mining_top_p),
                top_k=int(config.dpo_mining_top_k),
                max_new_tokens=int(config.dpo_mining_max_new_tokens),
                include_greedy=bool(config.dpo_mining_include_greedy),
                require_render=False,
            )
            gen_time_sum += (_now() - t_gen0)
            candidate_count = len(cands)
            consecutive_generate_failures = 0

            if reranker is not None and cands:
                stage = 'rerank'
                try:
                    best_r = choose_best_candidate(reranker, prompt, cands)
                    if best_r is not None and best_r not in cands:
                        cands.append(best_r)
                except Exception:
                    skip['reranker_exception'] += 1

            stage = 'candidate_sanitize'
            uniq = []
            seen = set()
            for c in cands:
                if not c:
                    continue
                key = _sha1(' '.join(str(c).split())[:20000])
                if key in seen:
                    skip['candidate_duplicate'] += 1
                    continue
                seen.add(key)
                c2 = _safe_sanitize_candidate(str(c), require_render=False)
                if c2 is None:
                    skip['candidate_sanitize_fail'] += 1
                    continue
                uniq.append(c2)

            if len(uniq) < 1:
                skip['no_valid_candidates'] += 1
                last_fail = 'no_valid_candidates'
                continue
            if len(uniq) < 2 and str(config.dpo_pair_strategy) == 'model_vs_model':
                skip['need_two_candidates_for_model_vs_model'] += 1
                last_fail = 'need_two_candidates_for_model_vs_model'
                continue

            stage = 'score'
            fast_scores, full_scores, dt_score = _score_candidates_two_stage(uniq, gt_svg)
            score_time_sum += dt_score
            full_count = len(full_scores)
            if not full_scores:
                skip['no_full_scores'] += 1
                last_fail = 'no_full_scores'
                continue

            stage = 'select_pair'
            pair = _select_pair(
                str(config.dpo_pair_strategy),
                gt_svg,
                full_scores,
                current_built=built,
                current_gt_anchor_built=gt_anchor_built,
            )
            if pair is None:
                last_fail = 'pair_not_selected'
                continue

            rows.append({
                'prompt': prompt_text,
                'chosen': pair['chosen'],
                'rejected': pair['rejected'],
                'chosen_score': float(pair['chosen_score']),
                'rejected_score': float(pair['rejected_score']),
                'pair_type': pair['pair_type'],
                'fast_best': float(fast_scores[0][0]) if fast_scores else None,
                'fast_worst': float(fast_scores[-1][0]) if fast_scores else None,
                'processed_index': int(i),
                'candidate_count': int(len(uniq)),
                'gt_margin': float(pair.get('gt_margin', pair['chosen_score'] - pair['rejected_score'])),
            })
            built += 1
            done_prompt_hash.add(prompt_key)
            recent_built[-1] = 1
            loop_ok = True
            last_fail = None
            last_error_short = None
            if pair['pair_type'] == 'gt_anchor':
                gt_anchor_built += 1
            else:
                model_model_built += 1

            if built % int(config.dpo_save_every_pairs) == 0:
                os.makedirs(os.path.dirname(config.dpo_pairs_path), exist_ok=True)
                pd.DataFrame(rows).to_parquet(config.dpo_pairs_path, index=False)
                print(f'[CKPT] saved {built} pairs -> {config.dpo_pairs_path}')

        except Exception as e:
            key = f'{stage}_exception'
            skip[key] += 1
            last_fail = key
            last_error_short = _short_error_text(e, limit=280)
            cuda_poisoned = cuda_poisoned or _looks_like_cuda_side_error(e)
            if stage == 'generate':
                consecutive_generate_failures += 1
            else:
                consecutive_generate_failures = 0

            should_print_err = (
                processed <= int(config.dpo_error_print_first_n)
                or (processed % int(config.dpo_error_print_every) == 0)
                or stage == 'generate'
            )
            if should_print_err:
                print(f'[DPO-ERROR] processed={processed} stage={stage} err={last_error_short}')

            _write_jsonl(config.dpo_bad_samples_path, {
                'kind': key,
                'processed_index': int(i),
                'stage': stage,
                'prompt': prompt,
                'candidate_count': int(candidate_count),
                'full_count': int(full_count),
                'error': repr(e),
                'error_short': last_error_short,
                'traceback': traceback.format_exc(limit=6),
            })

            if stage == 'generate' and int(config.dpo_abort_after_consecutive_generate_failures) > 0:
                if consecutive_generate_failures >= int(config.dpo_abort_after_consecutive_generate_failures):
                    raise RuntimeError(
                        f'Aborting DPO mining after {consecutive_generate_failures} consecutive generate failures. '
                        f'Last error: {last_error_short}'
                    ) from e

            if built == 0 and int(config.dpo_abort_if_zero_pairs_after_processed) > 0:
                if processed >= int(config.dpo_abort_if_zero_pairs_after_processed):
                    raise RuntimeError(
                        f'Aborting DPO mining because 0 pairs were built after {processed} processed prompts. '
                        f'Last failure={last_fail} last error={last_error_short}'
                    ) from e
        finally:
            should_log = (processed % int(config.dpo_log_every) == 0) or (processed <= int(config.dpo_step_debug_first_n))
            if should_log:
                elapsed = _now() - t_start
                prompts_per_sec = processed / max(elapsed, 1e-6)
                pairs_per_sec = built / max(elapsed, 1e-6)
                successful_gen_steps = max(processed - skip.get('generate_exception', 0), 1)
                avg_gen = gen_time_sum / successful_gen_steps
                avg_score = score_time_sum / max(processed, 1)
                remaining_pairs = max(0, int(config.dpo_target_pairs) - built)
                eta_pairs_sec = remaining_pairs / max(pairs_per_sec, 1e-6) if built > 0 else float('inf')
                recent_delta = int(sum(recent_built))
                eta_text = f'{eta_pairs_sec/60:.1f}m' if math.isfinite(eta_pairs_sec) else 'n/a'
                msg = (
                    f'[DPO-MINE] processed={processed}/{config.dpo_max_processed_prompts} '
                    f'built={built}/{config.dpo_target_pairs} '
                    f'gt_anchor={gt_anchor_built} model_vs_model={model_model_built} '
                    f'last_ok={loop_ok} last_fail={last_fail} stage={stage} '
                    f'cand={candidate_count} full={full_count} '
                    f'prompts/s={prompts_per_sec:.3f} pairs/s={pairs_per_sec:.3f} '
                    f'avg_gen={avg_gen:.2f}s avg_score={avg_score:.2f}s '
                    f'elapsed={elapsed/60:.1f}m ETA~{eta_text} '
                    f'recent{config.dpo_recent_window}={recent_delta}'
                )
                if last_error_short:
                    msg += f' | err={last_error_short}'
                if (_now() - last_gpu_poll) >= 30 or processed <= int(config.dpo_step_debug_first_n):
                    gs = _gpu_stats()
                    if gs:
                        msg += (f" | GPU util={gs['gpu_util_pct']:.0f}% "
                                f"VRAM={gs['mem_used_mb']:.0f}/{gs['mem_total_mb']:.0f}MB")
                    last_gpu_poll = _now()
                print(msg)

                if bool(config.dpo_trace_every_logs):
                    _write_jsonl(config.dpo_trace_path, {
                        'processed': int(processed),
                        'built': int(built),
                        'gt_anchor_built': int(gt_anchor_built),
                        'model_model_built': int(model_model_built),
                        'last_ok': bool(loop_ok),
                        'last_fail': last_fail,
                        'last_error_short': last_error_short,
                        'stage': stage,
                        'candidate_count': int(candidate_count),
                        'full_count': int(full_count),
                        'prompts_per_sec': float(prompts_per_sec),
                        'pairs_per_sec': float(pairs_per_sec),
                        'avg_gen_time': float(avg_gen),
                        'avg_score_time': float(avg_score),
                        'elapsed_sec': float(elapsed),
                        'eta_pairs_sec': None if not math.isfinite(eta_pairs_sec) else float(eta_pairs_sec),
                        'skip': dict(skip),
                    })

                if processed >= int(config.dpo_recent_window):
                    recent_delta = int(sum(recent_built))
                    if recent_delta < int(config.dpo_rate_collapse_threshold) and built < int(config.dpo_target_pairs):
                        print(
                            '[WARN] pair production rate is collapsing. '
                            'This warning is only meaningful if generation itself is succeeding. '
                            'If last_fail is generate_exception, fix model loading / adapter compatibility first.'
                        )

    os.makedirs(os.path.dirname(config.dpo_pairs_path), exist_ok=True)
    out_df = pd.DataFrame(rows)
    if len(out_df):
        out_df = out_df.drop_duplicates(subset=['prompt', 'chosen', 'rejected']).reset_index(drop=True)
        out_df = out_df[out_df['chosen'] != out_df['rejected']].reset_index(drop=True)
    out_df.to_parquet(config.dpo_pairs_path, index=False)
    print(f'[OK] DPO pairs saved -> {config.dpo_pairs_path} rows={len(out_df)} processed={processed}')
    print('[SKIP COUNTS]', dict(sorted(skip.items(), key=lambda x: (-x[1], x[0]))))

    try:
        del model
    except Exception:
        pass
    gc.collect()
    _safe_empty_cache(cuda_poisoned=cuda_poisoned)
    return out_df


# Backward-compatible alias
mine_dpo_pairs = mine_dpo_pairs_optimized

if RUN_BUILD_DPO:
    adapter_for_mining = CFG.sft_adapter_dir
    if bool(CFG.dpo_mining_use_dpo_adapter_if_present) and os.path.exists(CFG.dpo_adapter_dir):
        adapter_for_mining = CFG.dpo_adapter_dir
    if not os.path.exists(adapter_for_mining):
        raise FileNotFoundError('Need SFT adapter before mining DPO pairs.')
    dpo_df = mine_dpo_pairs(CFG, adapter_for_mining)


## 9. DPO Training

This stage trains on the mined preference pairs.

Notes:
- The mined dataset stays in TRL's `prompt / chosen / rejected` format.
- `pair_type` is preserved in the parquet for analysis, but the trainer only consumes the three DPO columns.
- If you used hybrid mining, the final training set can contain both `model_vs_model` and `gt_anchor` pairs.


In [ ]:
# =========================
# TRL / PEFT / Unsloth compatibility helpers
# Put this cell BEFORE run_dpo(...)
# =========================

def _ensure_warnings_issued_compat(model):
    """
    TRL 0.24's DPOTrainer expects model.warnings_issued to exist.
    Some Unsloth + PEFT wrapped models do not expose it, so we patch it onto
    the outer model and several common inner paths.
    Returns a list of patched object paths.
    """
    patched = []

    def _patch_obj(obj, path):
        if obj is None:
            return
        try:
            if not hasattr(obj, "warnings_issued") or getattr(obj, "warnings_issued") is None:
                setattr(obj, "warnings_issued", {})
                patched.append(path)
            elif not isinstance(getattr(obj, "warnings_issued"), dict):
                setattr(obj, "warnings_issued", dict(getattr(obj, "warnings_issued")))
                patched.append(path)
        except Exception:
            pass

    # outer
    _patch_obj(model, "model")

    # common PEFT / Unsloth paths
    try:
        _patch_obj(getattr(model, "base_model", None), "base_model")
    except Exception:
        pass

    try:
        bm = getattr(model, "base_model", None)
        _patch_obj(getattr(bm, "model", None), "base_model.model")
    except Exception:
        pass

    try:
        _patch_obj(getattr(model, "model", None), "model.model")
    except Exception:
        pass

    # deeper language-model path often seen in Qwen/Unsloth wrappers
    try:
        bm = getattr(model, "base_model", None)
        bm_model = getattr(bm, "model", None)
        lang_model = getattr(bm_model, "language_model", None)
        _patch_obj(lang_model, "base_model.model.language_model")
    except Exception:
        pass

    return patched

In [ ]:
def _pip_install_quiet(packages: List[str]):
    import sys, subprocess
    cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + list(packages)
    print('[PIP]', ' '.join(cmd))
    subprocess.check_call(cmd)

def _pip_uninstall_quiet(packages: List[str]):
    import sys, subprocess
    cmd = [sys.executable, '-m', 'pip', 'uninstall', '-y'] + list(packages)
    print('[PIP]', ' '.join(cmd))
    subprocess.call(cmd)

def _clear_modules_by_prefix(prefixes: List[str]):
    import sys
    doomed = []
    for name in list(sys.modules.keys()):
        if any(name == p or name.startswith(p + '.') for p in prefixes):
            doomed.append(name)
    for name in doomed:
        sys.modules.pop(name, None)
    if doomed:
        print(f'[DPO-DEPS] cleared modules: {sorted(doomed)[:8]}' + (' ...' if len(doomed) > 8 else ''))

def _patch_transformers_cache_symbol():
    import os
    try:
        import transformers.utils.hub as hub_mod
        if not hasattr(hub_mod, 'TRANSFORMERS_CACHE'):
            cache_dir = os.environ.get(
                'TRANSFORMERS_CACHE',
                os.path.expanduser('~/.cache/huggingface/transformers')
            )
            hub_mod.TRANSFORMERS_CACHE = cache_dir
            print(f'[DPO-DEPS] patched transformers.utils.hub.TRANSFORMERS_CACHE = {cache_dir}')
    except Exception as e:
        print(f'[DPO-DEPS] warning: failed to patch TRANSFORMERS_CACHE: {e}')

def _install_llm_blender_stub():
    import sys, types, importlib.machinery
    if 'llm_blender' in sys.modules:
        return
    mod = types.ModuleType('llm_blender')
    mod.__file__ = '<llm_blender_stub>'
    mod.__package__ = 'llm_blender'
    mod.__path__ = []
    mod.__spec__ = importlib.machinery.ModuleSpec('llm_blender', loader=None)

    class _DummyBlender:
        def __init__(self, *args, **kwargs):
            raise RuntimeError(
                'llm_blender stub was invoked. DPOTrainer itself does not need judges/llm_blender.'
            )

    mod.Blender = _DummyBlender
    sys.modules['llm_blender'] = mod
    print('[DPO-DEPS] injected llm_blender stub module')

def _install_weave_stub():
    import sys, types, importlib.machinery
    if 'weave' in sys.modules:
        return
    mod = types.ModuleType('weave')
    mod.__file__ = '<weave_stub>'
    mod.__package__ = 'weave'
    mod.__path__ = []
    mod.__spec__ = importlib.machinery.ModuleSpec('weave', loader=None)

    class _DummyEvaluationLogger:
        def __init__(self, *args, **kwargs):
            raise RuntimeError(
                'weave stub was invoked. DPOTrainer itself does not need weave callbacks.'
            )

    def _dummy_init(*args, **kwargs):
        raise RuntimeError('weave stub was invoked. This notebook does not use weave integrations.')

    mod.EvaluationLogger = _DummyEvaluationLogger
    mod.init = _dummy_init
    mod.finish = lambda *args, **kwargs: None
    sys.modules['weave'] = mod
    print('[DPO-DEPS] injected weave stub module')

def _patch_trl_optional_availability():
    """
    Robustly disable optional TRL integrations that are irrelevant for DPOTrainer in this notebook.
    This is stronger than only installing stub modules, because callbacks.py checks these helpers first.
    """
    import importlib
    trl_import_utils = importlib.import_module('trl.import_utils')
    patched = []
    if hasattr(trl_import_utils, 'is_llm_blender_available'):
        trl_import_utils.is_llm_blender_available = lambda: False
        patched.append('is_llm_blender_available=False')
    if hasattr(trl_import_utils, 'is_weave_available'):
        trl_import_utils.is_weave_available = lambda: False
        patched.append('is_weave_available=False')
    if patched:
        print('[DPO-DEPS] patched TRL optional availability:', ', '.join(patched))
    return trl_import_utils

def ensure_dpo_runtime_dependencies():
    import importlib

    need_mergekit = False
    try:
        import mergekit  # noqa: F401
    except ModuleNotFoundError:
        need_mergekit = True

    if need_mergekit:
        print('[DPO-DEPS] mergekit is missing. Installing it now...')
        _pip_install_quiet(['mergekit'])
        importlib.invalidate_caches()

    print('[DPO-DEPS] ensuring optional llm-blender / weave paths do not break DPO import ...')
    _pip_uninstall_quiet(['llm-blender', 'weave'])
    _clear_modules_by_prefix(['llm_blender', 'weave', 'trl'])
    _patch_transformers_cache_symbol()
    _install_llm_blender_stub()
    _install_weave_stub()
    importlib.invalidate_caches()

    try:
        _patch_trl_optional_availability()
        from trl import DPOConfig, DPOTrainer
        import trl
        import mergekit
        print(f'[DPO-DEPS] trl={getattr(trl, "__version__", "unknown")} | mergekit={getattr(mergekit, "__version__", "unknown")}')
        return DPOConfig, DPOTrainer
    except Exception as e:
        print(f'[DPO-DEPS] first import failed, retrying after hard clear ... err={repr(e)}')
        _clear_modules_by_prefix(['llm_blender', 'weave', 'trl'])
        _patch_transformers_cache_symbol()
        _install_llm_blender_stub()
        _install_weave_stub()
        importlib.invalidate_caches()
        _patch_trl_optional_availability()
        from trl import DPOConfig, DPOTrainer
        import trl
        import mergekit
        print(f'[DPO-DEPS] trl={getattr(trl, "__version__", "unknown")} | mergekit={getattr(mergekit, "__version__", "unknown")}')
        return DPOConfig, DPOTrainer


def _make_text_only_dpo_trainer_class(DPOTrainer):
    """
    TRL 0.24 may classify Qwen3.5-VL style models as vision models and route dataset prep
    through process_row(), which expects an `images` column. Our DPO dataset is text-only.
    This compatibility subclass falls back to tokenize_row() whenever no images are present.
    """
    class TextOnlyAwareDPOTrainer(DPOTrainer):
        @staticmethod
        def process_row(features, processing_class, max_prompt_length, max_completion_length, add_special_tokens):
            if ('images' not in features) or (features.get('images', None) is None):
                tok = _text_tokenizer(processing_class)
                return DPOTrainer.tokenize_row(
                    features,
                    tok,
                    max_prompt_length=max_prompt_length,
                    max_completion_length=max_completion_length,
                    add_special_tokens=add_special_tokens,
                )
            # Vision path only if images are truly present.
            return DPOTrainer.process_row(
                features,
                processing_class,
                max_prompt_length=max_prompt_length,
                max_completion_length=max_completion_length,
                add_special_tokens=add_special_tokens,
            )
    return TextOnlyAwareDPOTrainer

def _prepare_dpo_processing_class(tokenizer):
    # Always hand TRL a pure text tokenizer for this text-only SVG DPO dataset.
    tok = _text_tokenizer(tokenizer)
    try:
        tok.padding_side = 'left'
        if getattr(tok, 'pad_token', None) is None and getattr(tok, 'eos_token', None) is not None:
            tok.pad_token = tok.eos_token
    except Exception as e:
        print(f'[DPO] tokenizer padding setup warning: {e}')
    return tok

def run_dpo(config, sft_adapter_dir, dpo_pairs_path):
    DPOConfig, DPOTrainer = ensure_dpo_runtime_dependencies()
    import datasets as hf_datasets

    pairs_df = pd.read_parquet(dpo_pairs_path)
    if len(pairs_df) == 0:
        raise RuntimeError('DPO pairs parquet exists but has 0 rows.')

    print(f'[DPO] pair count: {len(pairs_df)}')
    if 'pair_type' in pairs_df.columns:
        print('[DPO] pair_type counts:')
        print(pairs_df['pair_type'].value_counts(dropna=False).to_dict())

    train_ds = hf_datasets.Dataset.from_pandas(
        pairs_df[['prompt', 'chosen', 'rejected']].reset_index(drop=True),
        preserve_index=False
    )

    model, tokenizer = load_trainable_model_from_sft_adapter(config, sft_adapter_dir)
    model.config.use_cache = False
    print(f'[DPO] model class={type(model).__name__} | inner={type(getattr(model, "base_model", None)).__name__}')

    patched_paths = _ensure_warnings_issued_compat(model)
    print(f'[DPO] patched warnings_issued on: {patched_paths}')

    processing_for_dpo = _prepare_dpo_processing_class(tokenizer)
    TextOnlyAwareDPOTrainer = _make_text_only_dpo_trainer_class(DPOTrainer)

    args = DPOConfig(
        output_dir=config.dpo_adapter_dir,
        per_device_train_batch_size=config.per_device_train_batch_size_dpo,
        gradient_accumulation_steps=config.gradient_accumulation_steps_dpo,
        learning_rate=config.learning_rate_dpo,
        num_train_epochs=config.num_train_epochs_dpo,
        logging_steps=config.logging_steps_dpo,
        save_steps=config.save_steps_dpo,
        save_total_limit=2,
        report_to=[],
        remove_unused_columns=False,
        bf16=torch.cuda.is_available(),
        fp16=False,
    )

    trainer = TextOnlyAwareDPOTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        processing_class=processing_for_dpo,
    )
    print('[DPO] training start ...')
    trainer.train()
    print('[DPO] training finished')

    os.makedirs(config.dpo_adapter_dir, exist_ok=True)
    trainer.model.save_pretrained(config.dpo_adapter_dir)
    try:
        _text_tokenizer(processing_for_dpo).save_pretrained(config.dpo_adapter_dir)
    except Exception:
        pass
    print(f'[OK] DPO adapter saved -> {config.dpo_adapter_dir}')

if RUN_DPO:
    if not os.path.exists(CFG.dpo_pairs_path):
        raise FileNotFoundError('Need DPO pairs.')
    run_dpo(CFG, CFG.sft_adapter_dir, CFG.dpo_pairs_path)


## 10. Build Submission

In [ ]:
import os, gc, time, math, json, hashlib
from typing import Optional, List, Dict, Any
import pandas as pd
import torch

def _sha1(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8", errors="ignore")).hexdigest()

def _simple_fallback_svg() -> str:
    return '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><rect width="256" height="256" fill="white"/></svg>'

def _normalize_prompt_key(prompt_text: str) -> str:
    return _sha1(" ".join(str(prompt_text).strip().split())[:4000])

def _submission_checkpoint_path(config) -> str:
    base = getattr(config, "submission_path", "submission.csv")
    root, ext = os.path.splitext(base)
    return root + ".partial.csv"

def _get_cfg(config, name: str, default):
    return getattr(config, name, default)

def _dynamic_submission_max_new_tokens(prompt_text: str, config) -> int:
    """
    Simple/short prompts usually do not need very long SVGs.
    This reduces average decode cost materially.
    """
    words = len(str(prompt_text).split())
    base = int(_get_cfg(config, "submission_max_new_tokens", 448))
    if words <= 5:
        return min(base, 320)
    if words <= 12:
        return min(base, 384)
    if words <= 24:
        return min(base, 448)
    return min(base, 576)

def _pick_best_submission_svg(cands: List[str]) -> Optional[str]:
    """
    Lightweight test-time selector without GT.
    Prefer compact, cleaner SVGs, with soft penalties on complexity.
    """
    if not cands:
        return None

    scored = []
    for s in cands:
        if not s:
            continue
        ln = len(s)
        n_path = s.count("<path")
        n_group = s.count("<g")
        n_text = s.count("<text")
        # Soft preference: compact + not overly path-heavy
        score = (
            -0.0018 * ln
            -0.0600 * n_path
            -0.0150 * n_group
            -0.0300 * n_text
        )
        scored.append((score, s))

    if not scored:
        return None
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][1]

def _safe_generate_candidates_submission(
    model,
    tokenizer,
    config,
    prompt_text: str,
    num_candidates: int,
    temperature: float,
    top_p: float,
    top_k: int,
    max_new_tokens: int,
):
    """
    Compatible wrapper around your existing generate_candidates().
    """
    try:
        return generate_candidates(
            model, tokenizer, config, prompt_text,
            num_candidates=num_candidates,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            max_new_tokens=max_new_tokens,
        )
    except TypeError:
        old = getattr(config, "inference_max_new_tokens", None)
        try:
            setattr(config, "inference_max_new_tokens", max_new_tokens)
            return generate_candidates(
                model, tokenizer, config, prompt_text,
                num_candidates=num_candidates,
                temperature=temperature,
                top_p=top_p,
                top_k=top_k,
            )
        finally:
            if old is not None:
                setattr(config, "inference_max_new_tokens", old)

def _sanitize_candidates_fast(cands: List[str], require_render: bool = False) -> List[str]:
    uniq = []
    seen = set()
    for c in cands or []:
        if not c:
            continue
        raw = str(c)
        key = _sha1(" ".join(raw.split())[:20000])
        if key in seen:
            continue
        seen.add(key)
        try:
            s = sanitize_svg(raw, require_render=require_render)
        except Exception:
            s = None
        if s is not None:
            uniq.append(s)
    return uniq

def generate_submission_svg_optimized(model, tokenizer, config, prompt_text: str) -> str:
    """
    Submission-time policy:
    1) fast main attempt: 1-2 candidates, no render-required sanitize
    2) conservative fallback: 1 candidate, render-required sanitize
    3) last resort fallback SVG
    """
    main_num_candidates = int(_get_cfg(config, "submission_num_candidates", 1))
    fallback_num_candidates = 1
    main_max_new_tokens = _dynamic_submission_max_new_tokens(prompt_text, config)

    # ---- Main attempt ----
    try:
        cands = _safe_generate_candidates_submission(
            model, tokenizer, config, prompt_text,
            num_candidates=main_num_candidates,
            temperature=float(_get_cfg(config, "submission_temperature", 0.65)),
            top_p=float(_get_cfg(config, "submission_top_p", 0.90)),
            top_k=int(_get_cfg(config, "submission_top_k", 40)),
            max_new_tokens=main_max_new_tokens,
        )
    except Exception:
        cands = []

    valid = _sanitize_candidates_fast(cands, require_render=False)
    best = _pick_best_submission_svg(valid)
    if best is not None:
        return best

    # ---- Conservative fallback ----
    try:
        cands2 = _safe_generate_candidates_submission(
            model, tokenizer, config, prompt_text,
            num_candidates=fallback_num_candidates,
            temperature=float(_get_cfg(config, "submission_fallback_temperature", 0.20)),
            top_p=float(_get_cfg(config, "submission_fallback_top_p", 0.80)),
            top_k=int(_get_cfg(config, "submission_fallback_top_k", 20)),
            max_new_tokens=min(320, main_max_new_tokens),
        )
    except Exception:
        cands2 = []

    valid2 = _sanitize_candidates_fast(cands2, require_render=True)
    if valid2:
        return valid2[0]

    return _simple_fallback_svg()

def build_submission_optimized(config, adapter_dir):
    # -------- knobs --------
    save_every = int(_get_cfg(config, "submission_save_every", 50))
    resume = bool(_get_cfg(config, "submission_resume", True))
    progress_every = int(_get_cfg(config, "submission_log_every", 10))
    checkpoint_path = _submission_checkpoint_path(config)

    # -------- model --------
    model, tokenizer = load_text_inference_model(config, adapter_dir=adapter_dir)
    try:
        model.eval()
    except Exception:
        pass

    # -------- data --------
    test_df = pd.read_csv(config.test_csv)
    total = len(test_df)

    # -------- resume --------
    rows: List[Dict[str, Any]] = []
    done_ids = set()
    prompt_cache: Dict[str, str] = {}
    fallback_count = 0

    if resume and os.path.exists(checkpoint_path):
        try:
            prev = pd.read_csv(checkpoint_path)
            rows = prev.to_dict("records")
            done_ids = set(prev["id"].tolist())
            if "prompt_hash" in prev.columns and "svg" in prev.columns:
                for _, r in prev[["prompt_hash", "svg"]].dropna().iterrows():
                    prompt_cache[str(r["prompt_hash"])] = str(r["svg"])
            print(f"[RESUME] loaded {len(rows)} rows from {checkpoint_path}")
        except Exception as e:
            print(f"[RESUME] failed to load partial checkpoint: {e}")

    t0 = time.monotonic()
    len_sum = 0
    processed_now = 0

    print("[SUBMIT] Entering inference loop.")

    with torch.inference_mode():
        for idx, row in enumerate(test_df.itertuples(index=False), start=1):
            sample_id = getattr(row, "id")
            if sample_id in done_ids:
                continue

            prompt = safe_text(getattr(row, "prompt"))
            pkey = _normalize_prompt_key(prompt)

            t_row0 = time.monotonic()

            if pkey in prompt_cache:
                svg = prompt_cache[pkey]
            else:
                svg = generate_submission_svg_optimized(model, tokenizer, config, prompt)
                prompt_cache[pkey] = svg

            is_fallback = (svg == _simple_fallback_svg())
            if is_fallback:
                fallback_count += 1

            rows.append({
                "id": sample_id,
                "svg": svg,
                "prompt_hash": pkey,   # temp field for resume cache; dropped in final save
            })
            done_ids.add(sample_id)

            row_time = time.monotonic() - t_row0
            processed_now += 1
            len_sum += len(svg)

            # periodic log
            if processed_now % progress_every == 0:
                elapsed = time.monotonic() - t0
                rate = processed_now / max(elapsed, 1e-6)
                remaining = total - len(done_ids)
                eta_min = remaining / max(rate, 1e-6) / 60.0
                avg_len = len_sum / max(processed_now, 1)
                print(
                    f"[SUBMIT] done={len(done_ids)}/{total} "
                    f"new_rate={rate:.3f} rows/s "
                    f"last={row_time:.1f}s "
                    f"avg_len={avg_len:.0f} "
                    f"fallbacks={fallback_count} "
                    f"ETA~{eta_min:.1f}m"
                )

            # periodic checkpoint
            if len(done_ids) % save_every == 0:
                tmp_df = pd.DataFrame(rows)
                os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)
                tmp_df.to_csv(checkpoint_path, index=False)
                print(f"[CKPT] saved partial -> {checkpoint_path} rows={len(tmp_df)}")

    # -------- final save --------
    sub_df = pd.DataFrame(rows)
    if "prompt_hash" in sub_df.columns:
        sub_df = sub_df.drop(columns=["prompt_hash"])

    os.makedirs(os.path.dirname(config.submission_path), exist_ok=True)
    sub_df.to_csv(config.submission_path, index=False)

    print(f"[OK] submission saved -> {config.submission_path}")
    print(f"[STATS] rows={len(sub_df)} avg_svg_len={sub_df['svg'].str.len().mean():.0f} max_svg_len={sub_df['svg'].str.len().max():.0f}")
    print(f"[STATS] fallbacks={fallback_count}/{len(sub_df)}")

    # cleanup
    try:
        del model
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return sub_df


# Recommended submission-time config knobs
CFG.submission_max_new_tokens = 448
CFG.submission_num_candidates = 1      # 1 is usually the best speed/quality tradeoff
CFG.submission_temperature = 0.65
CFG.submission_top_p = 0.90
CFG.submission_top_k = 40
CFG.submission_fallback_temperature = 0.20
CFG.submission_fallback_top_p = 0.80
CFG.submission_fallback_top_k = 20
CFG.submission_save_every = 50
CFG.submission_resume = True
CFG.submission_log_every = 10

if RUN_INFERENCE:
    final_adapter = CFG.dpo_adapter_dir if os.path.exists(CFG.dpo_adapter_dir) else CFG.sft_adapter_dir
    if not os.path.exists(final_adapter):
        raise FileNotFoundError("No adapter found for inference.")
    sub_df = build_submission_optimized(CFG, final_adapter)
    from google.colab import files
    files.download(CFG.submission_path)

In [ ]:
# def build_submission(config, adapter_dir):
#     test_df = pd.read_csv(config.test_csv)
#     model, tokenizer = load_text_inference_model(config, adapter_dir=adapter_dir)
#     reranker = LightReranker(config)
#     rows = []
#     t0 = time.time()
#     fallback_count = 0
#     print("Entering inference loop.")
#     for i, row in test_df.iterrows():
#         prompt = safe_text(row['prompt'])
#         t1 = time.time()
#         svg = generate_svg_final(model, tokenizer, reranker, config, prompt)
#         gen_time = time.time() - t1
#         is_fb = (svg == fallback_svg(prompt))
#         if is_fb:
#             fallback_count += 1
#         rows.append({'id': row['id'], 'svg': svg})
#         if (i + 1) % 10 == 0:
#             elapsed = time.time() - t0
#             eta = elapsed / (i + 1) * (len(test_df) - i - 1) / 60
#             print(f'  [{i+1}/{len(test_df)}] {gen_time:.1f}s | fb_total={fallback_count} | ETA={eta:.0f}min')

#     sub_df = pd.DataFrame(rows)
#     sub_df.to_csv(config.submission_path, index=False)
#     print(f'\n[OK] Submission saved -> {config.submission_path}')
#     print(f'SVG len mean={sub_df["svg"].str.len().mean():.0f} max={sub_df["svg"].str.len().max():.0f}')
#     print(f'Fallbacks: {fallback_count}/{len(sub_df)}')
#     return sub_df


# if RUN_INFERENCE:
#     final_adapter = CFG.dpo_adapter_dir if os.path.exists(CFG.dpo_adapter_dir) else CFG.sft_adapter_dir
#     if not os.path.exists(final_adapter):
#         raise FileNotFoundError('No adapter found for inference.')
#     sub_df = build_submission_fast(CFG, final_adapter)
#     from google.colab import files
#     files.download(CFG.submission_path)

## AI Tooling Disclosure

- **Claude (Anthropic)**: Coding assistance, debugging.